In [ ]:
# 0. 배치 실행 설정: 여기만 수정
ROI_LABEL = "Cheongna_international_city"
ROI_CENTER_LAT = 37.5279747
ROI_CENTER_LON = 126.6343758

# 끝일은 포함(inclusive) 기준으로 적는다.
BEFORE_START_DATE = "2003-01-01"
BEFORE_END_DATE = "2007-12-31"
AFTER_START_DATE = "2023-01-01"
AFTER_END_DATE = "2025-12-31"

# 계절별로 나눠 실행/저장한다. 필요한 계절만 남겨도 된다.
# spring=봄(3-5), summer=여름(6-8), autumn/fall=가을(9-11), winter=겨울(12,1,2)
SEASONS_TO_INCLUDE = ["봄", "여름", "가을", "겨울"]

# 실행 제어
EXECUTE_BATCH = True  #데이터 생성 여부
OVERWRITE = True      #기존 산출물 재사용 방지
STOP_ON_ERROR = False #에러 정지 여부


# ROI 기준 품질 필터: scene 전체 CLOUD_COVER가 아니라 패치 내부 cloud 비율로 거른다.
MIN_ROI_COVERAGE_FRACTION = 0.98  # scene footprint가 ROI 대부분을 덮어야 함
MAX_ROI_CLOUD_FRACTION = 0.30     # ROI 내부 cloud 비율 상한


In [ ]:
from pathlib import Path
import json
import hashlib
import math
import os
import re
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import ee
from pyproj import Transformer

# 0번 설정 셀 값을 사용한다.
ROI_LABEL = str(globals().get("ROI_LABEL", "songdo_international_city"))
ROI_CENTER_LAT = float(globals().get("ROI_CENTER_LAT", 37.3860959))
ROI_CENTER_LON = float(globals().get("ROI_CENTER_LON", 126.6389202))
BEFORE_START_DATE = str(globals().get("BEFORE_START_DATE", "2005-01-01"))
BEFORE_END_DATE = str(globals().get("BEFORE_END_DATE", "2007-12-31"))
AFTER_START_DATE = str(globals().get("AFTER_START_DATE", "2023-01-01"))
AFTER_END_DATE = str(globals().get("AFTER_END_DATE", "2025-12-31"))

SEASON_DEFINITIONS = {
    "spring": {"name_ko": "봄", "months": (3, 4, 5), "order": 0},
    "summer": {"name_ko": "여름", "months": (6, 7, 8), "order": 1},
    "autumn": {"name_ko": "가을", "months": (9, 10, 11), "order": 2},
    "winter": {"name_ko": "겨울", "months": (12, 1, 2), "order": 3},
}
SEASON_ALIASES = {
    "봄": "spring",
    "여름": "summer",
    "가을": "autumn",
    "겨울": "winter",
    "fall": "autumn",
}


def normalize_seasons(values):
    if values is None:
        values = list(SEASON_DEFINITIONS)
    if isinstance(values, str):
        values = [values]
    seasons = []
    for value in values:
        key = SEASON_ALIASES.get(str(value).strip().lower(), str(value).strip().lower())
        if key not in SEASON_DEFINITIONS:
            raise ValueError(f"Unknown season: {value}. Use one of {list(SEASON_DEFINITIONS)} or 봄/여름/가을/겨울.")
        if key not in seasons:
            seasons.append(key)
    return seasons


SEASONS_TO_INCLUDE = normalize_seasons(globals().get("SEASONS_TO_INCLUDE", list(SEASON_DEFINITIONS)))


def season_for_date(date_value):
    month = pd.Timestamp(date_value).month
    for key, definition in SEASON_DEFINITIONS.items():
        if month in definition["months"]:
            return key
    raise ValueError(f"No season configured for month={month}")


def ee_end_exclusive(date_ymd):
    return (pd.Timestamp(date_ymd) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


DATE_RANGES = [
    {"period_group": "before", "start": BEFORE_START_DATE, "end": ee_end_exclusive(BEFORE_END_DATE), "end_inclusive": BEFORE_END_DATE},
    {"period_group": "after", "start": AFTER_START_DATE, "end": ee_end_exclusive(AFTER_END_DATE), "end_inclusive": AFTER_END_DATE},
]

# Landsat 7은 제외한다.
LANDSAT_COLLECTIONS = [
    "LANDSAT/LT05/C02/T1_L2",
    "LANDSAT/LC08/C02/T1_L2",
    "LANDSAT/LC09/C02/T1_L2",
]
MAX_SCENE_CLOUD_COVER = 20
MIN_ROI_COVERAGE_FRACTION = float(globals().get("MIN_ROI_COVERAGE_FRACTION", 0.98))
MAX_ROI_CLOUD_FRACTION = float(globals().get("MAX_ROI_CLOUD_FRACTION", 0.20))
GRID_N = 256
PIXEL_SIZE_M = 60
CRS_BASE = "EPSG:32653"

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != "26.05_MLproj" and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent
if PROJECT_DIR.name != "26.05_MLproj":
    PROJECT_DIR = Path("/mnt/c/Users/shinh/eigenspace/Workspace/26.05_MLproj")

RECONST_DIR = PROJECT_DIR / "reconst"
BATCH_DIR = RECONST_DIR
ROI_BATCH_DIR = BATCH_DIR / ROI_LABEL
PERIOD_GROUPS = [item["period_group"] for item in DATE_RANGES]
PERIOD_DIRS = {group: ROI_BATCH_DIR / group for group in PERIOD_GROUPS}
SEASON_DIRS = {
    group: {season: PERIOD_DIRS[group] / season for season in SEASONS_TO_INCLUDE}
    for group in PERIOD_GROUPS
}
SEASON_RUNS_DIRS = {
    group: {season: SEASON_DIRS[group][season] / "runs" for season in SEASONS_TO_INCLUDE}
    for group in PERIOD_GROUPS
}
SCENE_MANIFEST_CSV = ROI_BATCH_DIR / f"{ROI_LABEL}_landsat_scene_manifest.csv"
EXECUTION_MANIFEST_CSV = ROI_BATCH_DIR / f"{ROI_LABEL}_batch_execution_manifest.csv"
RUNNER_ID = str(globals().get("RUNNER_ID") or f"runner_{os.getpid()}_{int(time.time())}")
RUNNER_ID_SAFE = re.sub(r"[^A-Za-z0-9_.-]+", "_", RUNNER_ID).strip("_") or "runner"
TASK_STATUS_DIR = ROI_BATCH_DIR / f"_task_status_{RUNNER_ID_SAFE}"
RUNNER_EXECUTION_MANIFEST_CSV = ROI_BATCH_DIR / f"{ROI_LABEL}_batch_execution_manifest_{RUNNER_ID_SAFE}.csv"
for p in [ROI_BATCH_DIR, TASK_STATUS_DIR] + [path for group_dirs in SEASON_RUNS_DIRS.values() for path in group_dirs.values()]:
    p.mkdir(parents=True, exist_ok=True)

# 실행 제어
EXECUTE_BATCH = bool(globals().get("EXECUTE_BATCH", True))
OVERWRITE = bool(globals().get("OVERWRITE", True))
STOP_ON_ERROR = bool(globals().get("STOP_ON_ERROR", False))
EE_PROJECT = os.environ.get("EE_PROJECT", "neat-acre-496003-i5")

print("batch dir:", BATCH_DIR)
print("roi batch dir:", ROI_BATCH_DIR)
print("runner id:", RUNNER_ID_SAFE)
print("runner mode: full ROI scene list")
print("runner status dir:", TASK_STATUS_DIR)
print("runner execution manifest:", RUNNER_EXECUTION_MANIFEST_CSV)
print("seasons:", {season: SEASON_DEFINITIONS[season]["name_ko"] for season in SEASONS_TO_INCLUDE})
print("period dirs:", PERIOD_DIRS)
print("season runs dirs:", SEASON_RUNS_DIRS)


In [ ]:
def run_token(value):
    return str(value).replace("/", "_").replace(chr(92), "_").replace(":", "_").replace(" ", "_").replace(".", "p")


def make_patch_grid(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):
    to_xy = Transformer.from_crs("EPSG:4326", crs_base, always_xy=True)
    cx, cy = to_xy.transform(float(center_lon), float(center_lat))
    half = grid_n * pixel_size_m / 2
    left, right = cx - half, cx + half
    bottom, top = cy - half, cy + half
    return {"crs": crs_base, "left": left, "right": right, "bottom": bottom, "top": top}


def ee_region_for_grid(grid):
    return ee.Geometry.Rectangle([grid["left"], grid["bottom"], grid["right"], grid["top"]], proj=grid["crs"], geodesic=False)


def init_ee():
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)


def merged_landsat_collection(region, start_date, end_date):
    parts = []
    for col_id in LANDSAT_COLLECTIONS:
        parts.append(
            ee.ImageCollection(col_id)
            .filterBounds(region)
            .filterDate(str(start_date), str(end_date))
            .filter(ee.Filter.lt("CLOUD_COVER", float(MAX_SCENE_CLOUD_COVER)))
        )
    col = parts[0]
    for part in parts[1:]:
        col = col.merge(part)
    return col.sort("system:time_start", True)


def qa_clear_mask(img):
    qa = img.select("QA_PIXEL")
    bad_bits = [0, 1, 2, 3, 4, 5]
    mask = ee.Image(1)
    for bit in bad_bits:
        mask = mask.And(qa.bitwiseAnd(1 << bit).eq(0))
    return mask


def landsat_band_map_from_product(product):
    if "LT05" in str(product):
        return {"thermal": "ST_B6"}
    return {"thermal": "ST_B10"}


def roi_scene_quality(scene_id, product):
    img = ee.Image(str(scene_id))
    thermal = landsat_band_map_from_product(product)["thermal"]
    data = img.select(thermal).gt(0)
    clear = qa_clear_mask(img).And(data)
    cloud = data.And(clear.Not())
    metrics = ee.Image.cat([
        data.rename("roi_coverage_fraction"),
        clear.rename("roi_clear_over_full_roi_fraction"),
        cloud.rename("roi_cloud_over_full_roi_fraction"),
    ]).unmask(0).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=PIXEL_SIZE_M,
        crs=CRS_BASE,
        maxPixels=1e8,
        bestEffort=True,
    ).getInfo()
    coverage = float(metrics.get("roi_coverage_fraction") or 0.0)
    clear_over_roi = float(metrics.get("roi_clear_over_full_roi_fraction") or 0.0)
    cloud_over_roi = float(metrics.get("roi_cloud_over_full_roi_fraction") or 0.0)
    roi_cloud_fraction = cloud_over_roi / coverage if coverage > 0 else 1.0
    roi_clear_fraction = clear_over_roi / coverage if coverage > 0 else 0.0
    return {
        "roi_coverage_fraction": coverage,
        "roi_cloud_fraction": float(roi_cloud_fraction),
        "roi_clear_fraction": float(roi_clear_fraction),
        "roi_cloud_over_full_roi_fraction": cloud_over_roi,
        "roi_clear_over_full_roi_fraction": clear_over_roi,
    }


def ee_scene_row(feature):
    props = feature["properties"]
    ts = pd.to_datetime(props["system:time_start"], unit="ms", utc=True).tz_convert("Asia/Seoul")
    product = props.get("LANDSAT_PRODUCT_ID") or feature["id"].split("/")[-1]
    return {
        "date": ts.strftime("%Y-%m-%d"),
        "timestamp_kst": ts.strftime("%Y-%m-%d %H:%M:%S%z"),
        "scene_id": feature["id"],
        "collection": "/".join(feature["id"].split("/")[:-1]),
        "landsat_product_id": product,
        "cloud_cover_scene": props.get("CLOUD_COVER"),
        "sun_azimuth": props.get("SUN_AZIMUTH"),
        "sun_elevation": props.get("SUN_ELEVATION"),
    }


init_ee()
grid = make_patch_grid(ROI_CENTER_LAT, ROI_CENTER_LON)
roi = ee_region_for_grid(grid)

rows = []
period_order = {item["period_group"]: i for i, item in enumerate(DATE_RANGES)}
for item in DATE_RANGES:
    group = item["period_group"]
    start = item["start"]
    end = item["end"]
    end_inclusive = item["end_inclusive"]
    col = merged_landsat_collection(roi, start, end)
    count = int(col.size().getInfo())
    print(f"{group}: {start} to {end_inclusive}: {count} scenes with CLOUD_COVER < {MAX_SCENE_CLOUD_COVER}")
    if count <= 0:
        continue
    info = col.toList(count).getInfo()
    for feature in info:
        row = ee_scene_row(feature)
        season = season_for_date(row["date"])
        if season not in SEASONS_TO_INCLUDE:
            continue
        row["period_group"] = group
        row["season_group"] = season
        row["season_name_ko"] = SEASON_DEFINITIONS[season]["name_ko"]
        row["period_start"] = start
        row["period_end"] = end
        row["period_end_inclusive"] = end_inclusive
        row["period_order"] = period_order[group]
        row["season_order"] = SEASON_DEFINITIONS[season]["order"]
        quality = roi_scene_quality(row["scene_id"], row["landsat_product_id"])
        row.update(quality)
        if row["roi_coverage_fraction"] < MIN_ROI_COVERAGE_FRACTION or row["roi_cloud_fraction"] > MAX_ROI_CLOUD_FRACTION:
            continue
        rows.append(row)

scene_manifest = pd.DataFrame(rows)
if scene_manifest.empty:
    raise RuntimeError("No matching Landsat scenes found for the configured ranges/ROI/cloud/season policy")
scene_manifest = scene_manifest.drop_duplicates("scene_id").sort_values(["period_order", "season_order", "date", "scene_id"]).reset_index(drop=True)

scene_manifest.insert(0, "batch_index", range(1, len(scene_manifest) + 1))
scene_manifest["roi_label"] = ROI_LABEL
scene_manifest["roi_center_lat"] = float(ROI_CENTER_LAT)
scene_manifest["roi_center_lon"] = float(ROI_CENTER_LON)
scene_manifest["max_scene_cloud_cover"] = float(MAX_SCENE_CLOUD_COVER)
scene_manifest["min_roi_coverage_fraction"] = float(MIN_ROI_COVERAGE_FRACTION)
scene_manifest["max_roi_cloud_fraction"] = float(MAX_ROI_CLOUD_FRACTION)
scene_manifest.to_csv(SCENE_MANIFEST_CSV, index=False, encoding="utf-8-sig")
print("scene manifest:", SCENE_MANIFEST_CSV)

season_count_rows = []
for period_group in PERIOD_GROUPS:
    for season_group in SEASONS_TO_INCLUDE:
        count = int(((scene_manifest["period_group"] == period_group) & (scene_manifest["season_group"] == season_group)).sum())
        season_count_rows.append({
            "period_group": period_group,
            "season_group": season_group,
            "season_name_ko": SEASON_DEFINITIONS[season_group]["name_ko"],
            "scene_count": count,
        })
season_counts = pd.DataFrame(season_count_rows)
print("전후/계절별 데이터 개수:")
print(season_counts.to_string(index=False))

summary_cols = ["batch_index", "period_group", "season_group", "season_name_ko", "date", "landsat_product_id", "cloud_cover_scene", "roi_coverage_fraction", "roi_cloud_fraction", "roi_clear_fraction", "scene_id"]
print(scene_manifest[summary_cols].to_string(index=False))


In [ ]:
# 내부 생성 로직: condition 생성과 CVAE 복원을 이 노트북 안에서 직접 실행한다.
_INTERNAL_CONDITION_GENERATION_CODE = 'from pathlib import Path\nfrom io import BytesIO\nfrom zipfile import ZipFile\nimport json, math, os, re, time, shutil\nfrom datetime import datetime\nfrom zoneinfo import ZoneInfo\n\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom PIL import Image, ImageDraw\nfrom pyproj import Transformer\nimport ee\n\ntry:\n    import rasterio\n    from rasterio.enums import Resampling\nexcept Exception:\n    rasterio = None\n    Resampling = None\n\n\n# ROI/날짜는 0번 설정 셀에서 바꾼다. 0번 셀을 건너뛰었을 때만 아래 기본값을 쓴다.\nTARGET_DATE = globals().get("TARGET_DATE", "2007-04-15")\nROI_LABEL = globals().get("ROI_LABEL", "manual_roi")\nROI_CENTER_LAT = globals().get("ROI_CENTER_LAT", 37.3860959)\nROI_CENTER_LON = globals().get("ROI_CENTER_LON", 126.6389202)\nPRESELECTED_LANDSAT_SCENE = globals().get("PRESELECTED_LANDSAT_SCENE", None)\n\n# 요청 날짜에 Landsat 장면이 없으면 요청일 이전/당일 중 가장 최근 장면을 쓴다.\nLANDSAT_LOOKBACK_DAYS = 3650\nMAX_SCENE_CLOUD_COVER = 20\nLANDSAT_COLLECTIONS = [\n    "LANDSAT/LT05/C02/T1_L2",\n    "LANDSAT/LC08/C02/T1_L2",\n    "LANDSAT/LC09/C02/T1_L2",\n]\n\n# EGIS unknown 처리는 fill/save 셀 실행 시 입력받는다. 1=바다, 2=도심\nOVERWRITE = bool(globals().get("OVERWRITE", True))\n\n# 외부 API 설정: 환경변수가 없으면 고정 키를 바로 사용한다.\nEE_PROJECT = os.environ.get("EE_PROJECT", "neat-acre-496003-i5")\nDEFAULT_KMA_AUTH_KEY = "34j6vBWrSGiI-rwVq0hoRg"\nKMA_AUTH_KEY = os.environ.get("KMA_AUTH_KEY") or os.environ.get("KMA_APIHUB_AUTH_KEY") or DEFAULT_KMA_AUTH_KEY\nREQUEST_TIMEOUT = 180\n\n\nPROJECT_DIR = Path.cwd().resolve()\nwhile PROJECT_DIR.name != "26.05_MLproj" and PROJECT_DIR.parent != PROJECT_DIR:\n    PROJECT_DIR = PROJECT_DIR.parent\nif PROJECT_DIR.name != "26.05_MLproj":\n    PROJECT_DIR = Path("/mnt/c/Users/shinh/eigenspace/Workspace/26.05_MLproj")\n\nRECONST_DIR = PROJECT_DIR / "reconst"\nRUNS_ROOT_BASE = Path(globals().get("RUNS_ROOT_DIR") or (RECONST_DIR / "runs"))\n\n\ndef run_token(value):\n    return str(value).replace("/", "_").replace(chr(92), "_").replace(":", "_").replace(" ", "_").replace(".", "p")\n\n\nREQUESTED_DATE_LABEL = pd.Timestamp(TARGET_DATE).strftime("%Y-%m-%d")\nRUN_LABEL = None\nRUN_ROOT_DIR = None\nCONDITION_DIR = None\nRECONSTRUCTION_DIR = None\nOUT_DIR = None\nVIS_DIR = None\nSCRATCH_DIR = None\nREVIEW_DIR = None\nDIRECT_RAW_DIR = None\nREVIEW_CSV = None\nDECISION_CSV = None\n\nGRID_N = 256\nPIXEL_SIZE_M = 60\nCRS_BASE = "EPSG:32653"\nNODATA = -9999.0\nHOUR_KST = "1100"\nLST_VALID_MIN_K = 150.0\nLST_VALID_MAX_K = 380.0\nALBEDO_FALLBACK_VALUE = 0.18\nALBEDO_FILL_RADII_PX = (1, 2, 4, 8, 16, 32, 64, 128)\nTERRAIN_BANDS = ["elevation", "slope", "aspect_sin", "aspect_cos", "hillshade_landsat"]\nALBEDO_BANDS = ["albedo"]\nAWS_BANDS = ["avg_temp", "wind_u", "wind_v", "rain_1h", "humidity"]\nMETA_CHANNELS = ["day_sin", "day_cos", "sun_azimuth_norm", "sun_elevation_norm", "lat_norm", "lon_norm"]\nCONDITION_CHANNELS = TERRAIN_BANDS + ALBEDO_BANDS + AWS_BANDS + [\n    "egis_cat_110_urban_residential", "egis_cat_120_urban_industrial", "egis_cat_130_urban_commercial",\n    "egis_cat_140_urban_cultural_public", "egis_cat_150_urban_transportation", "egis_cat_160_urban_mixed",\n    "egis_cat_200_agriculture", "egis_cat_300_forest", "egis_cat_400_grassland", "egis_cat_500_wetland",\n    "egis_cat_600_barren", "egis_cat_710_water_inland", "egis_cat_720_water_marine",\n]\n\nMETA_NORM_SOURCE = "reconst_fixed_training_meta_scale"\nLAT_META_MEAN = 36.22751454452666\nLAT_META_STD = 0.974472059049708\nLON_META_MEAN = 127.70786578208728\nLON_META_STD = 0.8830399708103981\n\n\nprint("requested target_date:", REQUESTED_DATE_LABEL)\nprint("data source: direct EE + KMA + EGIS WMS")\nprint("meta norm source:", META_NORM_SOURCE)\nprint("lat/lon meta norm:", {"lat_mean": LAT_META_MEAN, "lat_std": LAT_META_STD, "lon_mean": LON_META_MEAN, "lon_std": LON_META_STD})\n\n\ndef make_patch_grid(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):\n    to_xy = Transformer.from_crs("EPSG:4326", crs_base, always_xy=True)\n    cx, cy = to_xy.transform(float(center_lon), float(center_lat))\n    half = grid_n * pixel_size_m / 2\n    left, right = cx - half, cx + half\n    bottom, top = cy - half, cy + half\n    xs = left + (np.arange(grid_n) + 0.5) * pixel_size_m\n    ys = top - (np.arange(grid_n) + 0.5) * pixel_size_m\n    xx, yy = np.meshgrid(xs, ys)\n    return {"n": grid_n, "pixel_size": pixel_size_m, "crs": crs_base, "left": left, "right": right, "bottom": bottom, "top": top, "xx": xx.astype(np.float32), "yy": yy.astype(np.float32)}\n\ndef ee_region_for_grid(grid):\n    return ee.Geometry.Rectangle([grid["left"], grid["bottom"], grid["right"], grid["top"]], proj=grid["crs"], geodesic=False)\n\ndef validate_kma_auth_key(key):\n    key = str(key).strip()\n    if len(key) < 10 or "°" in key:\n        raise ValueError("KMA APIHub authKey가 비어 있거나 잘못되었습니다.")\n    return key\n\ndef initialize_external_clients():\n    try:\n        ee.Initialize(project=EE_PROJECT)\n    except Exception:\n        ee.Authenticate()\n        ee.Initialize(project=EE_PROJECT)\n    global KMA_AUTH_KEY\n    if not KMA_AUTH_KEY:\n        raise ValueError("KMA APIHub authKey가 없습니다. DEFAULT_KMA_AUTH_KEY 또는 환경변수를 확인하세요.")\n    KMA_AUTH_KEY = validate_kma_auth_key(KMA_AUTH_KEY)\n    print("EE initialized; KMA key accepted")\n\ninitialize_external_clients()\ngrid = make_patch_grid(ROI_CENTER_LAT, ROI_CENTER_LON)\nroi = ee_region_for_grid(grid)\n\n\ndef merged_landsat_collection(roi, start_date, end_date):\n    parts = []\n    for col_id in LANDSAT_COLLECTIONS:\n        parts.append(\n            ee.ImageCollection(col_id)\n            .filterBounds(roi)\n            .filterDate(str(start_date), str(end_date))\n            .filter(ee.Filter.lt("CLOUD_COVER", float(MAX_SCENE_CLOUD_COVER)))\n        )\n    col = parts[0]\n    for part in parts[1:]:\n        col = col.merge(part)\n    return col\n\ndef ee_scene_row(feature):\n    props = feature["properties"]\n    ts = pd.to_datetime(props["system:time_start"], unit="ms", utc=True).tz_convert("Asia/Seoul")\n    return {\n        "date": ts.strftime("%Y-%m-%d"),\n        "timestamp_kst": ts.strftime("%Y-%m-%d %H:%M:%S%z"),\n        "scene_id": feature["id"],\n        "collection": "/".join(feature["id"].split("/")[:-1]),\n        "landsat_product_id": props.get("LANDSAT_PRODUCT_ID") or feature["id"].split("/")[-1],\n        "cloud_cover_scene": props.get("CLOUD_COVER"),\n        "sun_azimuth": props.get("SUN_AZIMUTH"),\n        "sun_elevation": props.get("SUN_ELEVATION"),\n    }\n\ndef select_landsat_scene():\n    requested = pd.Timestamp(TARGET_DATE)\n    exact_start = requested.strftime("%Y-%m-%d")\n    exact_end = (requested + pd.Timedelta(days=1)).strftime("%Y-%m-%d")\n    recent_start = (requested - pd.Timedelta(days=int(LANDSAT_LOOKBACK_DAYS))).strftime("%Y-%m-%d")\n    recent_end = exact_end\n    exact_col = merged_landsat_collection(roi, exact_start, exact_end)\n    exact_count = int(exact_col.size().getInfo())\n    recent_col = merged_landsat_collection(roi, recent_start, recent_end).sort("system:time_start", False)\n    recent_count = int(recent_col.size().getInfo())\n    if recent_count <= 0:\n        raise RuntimeError(f"{TARGET_DATE} 이전 {LANDSAT_LOOKBACK_DAYS}일 내 ROI와 겹치는 Landsat 장면이 없습니다.")\n    recent_info = recent_col.limit(50).getInfo()\n    recent_rows = [ee_scene_row(feature) for feature in recent_info["features"]]\n    selected = recent_rows[0]\n    selected_date_count = sum(1 for row in recent_rows if row["date"] == selected["date"])\n    exact_rows = []\n    if exact_count:\n        exact_rows = [ee_scene_row(feature) for feature in exact_col.limit(20).getInfo()["features"]]\n    return {\n        "requested_date": requested.strftime("%Y-%m-%d"),\n        "landsat_exact_available": bool(exact_count > 0),\n        "landsat_exact_count": int(exact_count),\n        "selected_landsat_date": selected["date"],\n        "selected_landsat_date_count": int(selected_date_count),\n        "selected_landsat_scene_id": selected["scene_id"],\n        "selected_landsat_product_id": selected["landsat_product_id"],\n        "selected_landsat_collection": selected["collection"],\n        "selected_landsat_cloud_cover_scene": selected["cloud_cover_scene"],\n        "selected_landsat_sun_azimuth": selected["sun_azimuth"],\n        "selected_landsat_sun_elevation": selected["sun_elevation"],\n        "exact_landsat_scenes": exact_rows,\n        "recent_landsat_scenes_checked": recent_rows,\n    }\n\ndef landsat_info_from_scene_row(scene_row, requested_date=None):\n    selected_date = str(scene_row["date"])\n    requested = pd.Timestamp(requested_date or selected_date).strftime("%Y-%m-%d")\n    return {\n        "requested_date": requested,\n        "landsat_exact_available": requested == selected_date,\n        "landsat_exact_count": int(1 if requested == selected_date else 0),\n        "selected_landsat_date": selected_date,\n        "selected_landsat_date_count": int(scene_row.get("selected_landsat_date_count", 1)),\n        "selected_landsat_scene_id": str(scene_row["scene_id"]),\n        "selected_landsat_product_id": str(scene_row["landsat_product_id"]),\n        "selected_landsat_collection": str(scene_row["collection"]),\n        "selected_landsat_cloud_cover_scene": scene_row.get("cloud_cover_scene"),\n        "selected_landsat_sun_azimuth": scene_row.get("sun_azimuth"),\n        "selected_landsat_sun_elevation": scene_row.get("sun_elevation"),\n        "exact_landsat_scenes": [dict(scene_row)] if requested == selected_date else [],\n        "recent_landsat_scenes_checked": [dict(scene_row)],\n    }\n\nlandsat_info = landsat_info_from_scene_row(PRESELECTED_LANDSAT_SCENE, TARGET_DATE) if PRESELECTED_LANDSAT_SCENE else select_landsat_scene()\nselected_date = pd.Timestamp(landsat_info["selected_landsat_date"])\nanalysis_year = int(selected_date.year)\nrow = {\n    "date": landsat_info["selected_landsat_date"],\n    "year": analysis_year,\n    "latitude": float(ROI_CENTER_LAT),\n    "longitude": float(ROI_CENTER_LON),\n    "scene_id": landsat_info["selected_landsat_scene_id"],\n    "collection": landsat_info["selected_landsat_collection"],\n    "landsat_product_id": landsat_info["selected_landsat_product_id"],\n    "sun_azimuth": float(landsat_info["selected_landsat_sun_azimuth"]),\n    "sun_elevation": float(landsat_info["selected_landsat_sun_elevation"]),\n}\nprint("requested_date:", landsat_info["requested_date"])\nprint("landsat_exact_available:", landsat_info["landsat_exact_available"], "count:", landsat_info["landsat_exact_count"])\nprint("selected_landsat_date:", landsat_info["selected_landsat_date"])\nprint("selected_landsat_product_id:", landsat_info["selected_landsat_product_id"])\nprint("selected_landsat_scene_id:", landsat_info["selected_landsat_scene_id"])\n\nRUN_DATE_LABEL = landsat_info["selected_landsat_date"]\nRUN_PRODUCT_LABEL = landsat_info["selected_landsat_product_id"]\nRUN_LABEL = f"{run_token(RUN_DATE_LABEL)}_{run_token(RUN_PRODUCT_LABEL)}_{run_token(ROI_LABEL)}_lat{run_token(round(float(ROI_CENTER_LAT), 5))}_lon{run_token(round(float(ROI_CENTER_LON), 5))}"\nRUN_ROOT_DIR = RUNS_ROOT_BASE / RUN_LABEL\nCONDITION_DIR = RUN_ROOT_DIR / "outputs" / "condition"\nRECONSTRUCTION_DIR = RUN_ROOT_DIR / "outputs" / "reconstruction"\nOUT_DIR = CONDITION_DIR\nVIS_DIR = RUN_ROOT_DIR / "outputs" / "visualization"\nSCRATCH_DIR = RUN_ROOT_DIR / "_scratch"\nREVIEW_DIR = SCRATCH_DIR / "unknown_review"\nDIRECT_RAW_DIR = SCRATCH_DIR / "direct_raw"\nREVIEW_CSV = REVIEW_DIR / "egis_unknown_review.csv"\nDECISION_CSV = REVIEW_DIR / "egis_unknown_decisions.csv"\nif OVERWRITE and RUN_ROOT_DIR.exists():\n    shutil.rmtree(RUN_ROOT_DIR)\nfor p in [CONDITION_DIR, RECONSTRUCTION_DIR, VIS_DIR, REVIEW_DIR, DIRECT_RAW_DIR]:\n    p.mkdir(parents=True, exist_ok=True)\nprint("run_label:", RUN_LABEL)\nprint("run root:", RUN_ROOT_DIR)\nprint("condition output:", CONDITION_DIR)\nprint("visualization output:", VIS_DIR)\nprint("scratch:", SCRATCH_DIR)\n\n\ndef ee_stack(image, bands, grid):\n    if rasterio is None:\n        raise ImportError("rasterio가 필요합니다. 현재 환경에 설치 후 이 노트북을 실행하세요.")\n    image = image.select(bands).toFloat().unmask(NODATA)\n    transform = [grid["pixel_size"], 0, grid["left"], 0, -grid["pixel_size"], grid["top"]]\n    params = {"name": "patch", "bands": bands, "crs": grid["crs"], "crs_transform": transform, "dimensions": [grid["n"], grid["n"]], "filePerBand": False, "format": "GEO_TIFF"}\n    url = image.getDownloadURL(params)\n    r = requests.get(url, timeout=300)\n    r.raise_for_status()\n    data = r.content\n    if data.startswith(b"PK"):\n        with ZipFile(BytesIO(data)) as zf:\n            names = [n for n in zf.namelist() if n.lower().endswith((".tif", ".tiff"))]\n            if not names:\n                raise RuntimeError("EE zip has no tif")\n            data = zf.read(names[0])\n    with rasterio.open(BytesIO(data)) as ds:\n        arr = ds.read(out_shape=(len(bands), grid["n"], grid["n"]), resampling=Resampling.nearest).astype(np.float32)\n    arr[arr <= NODATA / 2] = np.nan\n    return arr\n\ndef landsat_image(row):\n    return ee.Image(str(row["scene_id"]))\n\ndef landsat_band_map(row):\n    scene_ref = " ".join(str(row.get(key, "")) for key in ["scene_id", "collection", "landsat_product_id", "selected_landsat_scene_id"])\n    if "LT05" in scene_ref:\n        return {\n            "thermal": "ST_B6",\n            "blue": "SR_B1",\n            "red": "SR_B3",\n            "nir": "SR_B4",\n            "swir1": "SR_B5",\n            "swir2": "SR_B7",\n        }\n    return {\n        "thermal": "ST_B10",\n        "blue": "SR_B2",\n        "red": "SR_B4",\n        "nir": "SR_B5",\n        "swir1": "SR_B6",\n        "swir2": "SR_B7",\n    }\n\ndef qa_clear_mask(img):\n    qa = img.select("QA_PIXEL")\n    bad_bits = [0, 1, 2, 3, 4, 5]\n    mask = ee.Image(1)\n    for bit in bad_bits:\n        mask = mask.And(qa.bitwiseAnd(1 << bit).eq(0))\n    return mask\n\n\ndef landsat_lst_direct(row, grid):\n    img = landsat_image(row)\n    lst_band = landsat_band_map(row)["thermal"]\n    lst = img.select(lst_band).multiply(0.00341802).add(149.0).rename("lst_k_raw")\n    data = img.select(lst_band).gt(0)\n    clear = qa_clear_mask(img).And(data).rename("clear_mask")\n    cloud = data.And(clear.Not()).rename("cloud_fill_mask")\n    stack = ee_stack(ee.Image.cat([lst.updateMask(clear), clear.toFloat(), cloud.toFloat()]), ["lst_k_raw", "clear_mask", "cloud_fill_mask"], grid)\n    lst_raw = stack[0].astype(np.float32)\n    clear_mask = (stack[1] > 0.5)\n    cloud_fill_mask = (stack[2] > 0.5)\n    invalid = ~np.isfinite(lst_raw) | (lst_raw < LST_VALID_MIN_K) | (lst_raw > LST_VALID_MAX_K)\n    lst_raw[invalid] = np.nan\n    clear_mask = clear_mask & np.isfinite(lst_raw)\n    return lst_raw.astype(np.float32), clear_mask.astype(np.uint8), cloud_fill_mask.astype(np.uint8)\n\ndef terrain_direct(row, grid):\n    dem = ee.Image("USGS/SRTMGL1_003").select("elevation").resample("bilinear")\n    terrain = ee.Terrain.products(dem)\n    slope = terrain.select("slope")\n    aspect_rad = terrain.select("aspect").multiply(np.pi / 180.0)\n    aspect_sin = aspect_rad.sin().rename("aspect_sin")\n    aspect_cos = aspect_rad.cos().rename("aspect_cos")\n    hillshade = ee.Terrain.hillshade(dem, float(row["sun_azimuth"]), float(row["sun_elevation"])).rename("hillshade_landsat")\n    img = ee.Image.cat([dem.rename("elevation"), slope.rename("slope"), aspect_sin, aspect_cos, hillshade])\n    return ee_stack(img, TERRAIN_BANDS, grid).astype(np.float32)\n\ndef fill_nan_by_expanding_mean_2d(arr, radii=ALBEDO_FILL_RADII_PX, fallback_value=ALBEDO_FALLBACK_VALUE):\n    filled = np.asarray(arr, dtype=np.float32).copy()\n    nan_before = int((~np.isfinite(filled)).sum())\n    radius_used = np.zeros(filled.shape, dtype=np.int16)\n    for radius in radii:\n        pending = ~np.isfinite(filled)\n        if not bool(pending.any()):\n            break\n        updates = []\n        ys, xs = np.where(pending)\n        for y, x in zip(ys, xs):\n            y0, y1 = max(0, int(y) - int(radius)), min(filled.shape[0], int(y) + int(radius) + 1)\n            x0, x1 = max(0, int(x) - int(radius)), min(filled.shape[1], int(x) + int(radius) + 1)\n            window = filled[y0:y1, x0:x1]\n            valid = np.isfinite(window)\n            if bool(valid.any()):\n                updates.append((int(y), int(x), float(np.nanmean(window[valid]))))\n        for y, x, value in updates:\n            filled[y, x] = value\n            radius_used[y, x] = int(radius)\n    pending = ~np.isfinite(filled)\n    fallback_pixels = int(pending.sum())\n    if fallback_pixels:\n        fallback = float(np.nanmedian(filled)) if bool(np.isfinite(filled).any()) else float(fallback_value)\n        filled[pending] = fallback\n    nan_after = int((~np.isfinite(filled)).sum())\n    return filled.astype(np.float32), {\n        "nan_before": nan_before,\n        "nan_after": nan_after,\n        "local_fill_pixels": int(nan_before - fallback_pixels),\n        "fallback_fill_pixels": fallback_pixels,\n        "max_radius_px": int(radius_used.max()) if radius_used.size else 0,\n    }\n\n\ndef fill_albedo_nan_stack(arr):\n    filled = np.asarray(arr, dtype=np.float32).copy()\n    per_band = []\n    for band_idx in range(filled.shape[0]):\n        filled[band_idx], stats = fill_nan_by_expanding_mean_2d(filled[band_idx])\n        per_band.append(stats)\n    return filled.astype(np.float32), {\n        "albedo_fill_method": "attempt3_clear_mask_focal_mean_plus_local_expanding_mean",\n        "albedo_nan_pixels_before": int(sum(item["nan_before"] for item in per_band)),\n        "albedo_nan_pixels_after": int(sum(item["nan_after"] for item in per_band)),\n        "albedo_local_fill_pixels": int(sum(item["local_fill_pixels"] for item in per_band)),\n        "albedo_fallback_fill_pixels": int(sum(item["fallback_fill_pixels"] for item in per_band)),\n        "albedo_fill_max_radius_px": int(max([item["max_radius_px"] for item in per_band] or [0])),\n        "albedo_fallback_value": float(ALBEDO_FALLBACK_VALUE),\n    }\n\n\ndef albedo_direct(row, grid):\n    img = landsat_image(row)\n    band_map = landsat_band_map(row)\n    clear_img = img.updateMask(qa_clear_mask(img))\n    def sr(name):\n        return clear_img.select(name).multiply(0.0000275).add(-0.2)\n    albedo = sr(band_map["blue"]).multiply(0.356).add(sr(band_map["red"]).multiply(0.130)).add(sr(band_map["nir"]).multiply(0.373)).add(sr(band_map["swir1"]).multiply(0.085)).add(sr(band_map["swir2"]).multiply(0.072)).subtract(0.0018).rename("albedo")\n    # attempt3 생성법: clear pixel albedo를 만들고 focal mean으로 구름/결측을 먼저 보완한다.\n    albedo_attempt3 = albedo.unmask(albedo.focal_mean(radius=5, units="pixels", iterations=2)).rename("albedo")\n    # 해안/도서처럼 EE focal mean만으로 남는 빈칸은 넓은 focal mean과 로컬 확장 평균으로 끝까지 채운다.\n    albedo_attempt3 = albedo_attempt3.unmask(albedo_attempt3.focal_mean(radius=16, units="pixels", iterations=1)).rename("albedo")\n    arr = ee_stack(albedo_attempt3, ALBEDO_BANDS, grid).astype(np.float32)\n    global albedo_fill_stats\n    arr, albedo_fill_stats = fill_albedo_nan_stack(arr)\n    return arr.astype(np.float32)\n\nlst_k_raw_direct, lst_clear_mask_direct, lst_cloud_fill_mask_direct = landsat_lst_direct(row, grid)\nterrain = terrain_direct(row, grid)\nalbedo = albedo_direct(row, grid)\nprint("lst raw:", lst_k_raw_direct.shape, "clear pixels:", int(lst_clear_mask_direct.sum()))\nprint("terrain:", terrain.shape, "albedo:", albedo.shape)\nprint("albedo fill stats:", albedo_fill_stats)\n\n\nAWSH_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/awsh.php"\nSTN_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"\nKMA_RETRIES = 5\nKMA_BACKOFF_SECONDS = 3\nAWSH_COLUMNS = {"TA": ["TM", "STN", "TA"], "WD": ["TM", "STN", "WD"], "WS": ["TM", "STN", "WS"], "RN": ["TM", "STN", "RN"], "HM": ["TM", "STN", "HM"]}\nAWS_VALUE_RENAME = {"TA": "avg_temp", "RN": "rain_1h", "HM": "humidity"}\n\ndef decode_kma_response(content):\n    for enc in ["utf-8", "cp949", "euc-kr"]:\n        try:\n            return content.decode(enc)\n        except UnicodeDecodeError:\n            continue\n    return content.decode("utf-8", errors="replace")\n\ndef valid_data_lines(text):\n    return [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.startswith("#")]\n\ndef request_kma_with_retry(url, params, timeout=60):\n    last_exc = None\n    for attempt in range(1, KMA_RETRIES + 1):\n        try:\n            r = requests.get(url, params=params, timeout=timeout)\n            r.raise_for_status()\n            return r\n        except Exception as exc:\n            last_exc = exc\n            wait = KMA_BACKOFF_SECONDS * attempt\n            print(f"[KMA retry {attempt}/{KMA_RETRIES}] {params} -> {type(exc).__name__}: {exc}; wait {wait}s")\n            time.sleep(wait)\n    raise RuntimeError(f"KMA request failed after retries: {params}; last={last_exc}")\n\ndef fetch_awsh_var(var, tm, auth_key):\n    params = {"var": var, "tm": tm, "stn": "0", "help": "0", "authKey": auth_key}\n    r = request_kma_with_retry(AWSH_BASE_URL, params, timeout=60)\n    lines = valid_data_lines(decode_kma_response(r.content))\n    rows = [re.split(r"\\s+", ln)[: len(AWSH_COLUMNS[var])] for ln in lines]\n    df = pd.DataFrame(rows, columns=AWSH_COLUMNS[var]) if rows else pd.DataFrame(columns=AWSH_COLUMNS[var])\n    for col in df.columns:\n        if col != "TM":\n            df[col] = pd.to_numeric(df[col], errors="coerce")\n    df = df.dropna(subset=["STN"])\n    df["STN"] = df["STN"].astype(int)\n    return df\n\ndef parse_station_info_text(text):\n    rows = []\n    for ln in valid_data_lines(text):\n        parts = re.split(r"\\s+", ln)\n        nums = [x for x in parts if re.fullmatch(r"-?\\d+(\\.\\d+)?", x)]\n        if len(nums) >= 3:\n            stn = int(float(nums[0]))\n            lon = float(nums[1])\n            lat = float(nums[2])\n            if 120.0 <= lon <= 135.0 and 30.0 <= lat <= 45.0:\n                rows.append({"STN": stn, "LON": lon, "LAT": lat})\n    return pd.DataFrame(rows, columns=["STN", "LON", "LAT"])\n\n\ndef fetch_station_info(auth_key, tm):\n    # KMA가 stn="" 조합에서 500을 반환하는 경우가 있어 stn=0/help=0을 먼저 시도한다.\n    variants = [\n        {"inf": "AWS", "stn": "0", "tm": tm, "help": "0", "authKey": auth_key},\n        {"inf": "AWS", "stn": "0", "tm": tm, "help": "1", "authKey": auth_key},\n        {"inf": "AWS", "stn": "", "tm": tm, "help": "0", "authKey": auth_key},\n        {"inf": "AWS", "stn": "", "tm": tm, "help": "1", "authKey": auth_key},\n    ]\n    errors = []\n    for params in variants:\n        try:\n            r = request_kma_with_retry(STN_BASE_URL, params, timeout=60)\n            text = decode_kma_response(r.content)\n            df = parse_station_info_text(text)\n            if not df.empty:\n                print(f"[KMA station info] {len(df)} AWS stations via stn={params[\'stn\']!r} help={params[\'help\']}")\n                return df.drop_duplicates("STN")\n            errors.append(f"stn={params[\'stn\']!r} help={params[\'help\']}: empty parse")\n        except Exception as exc:\n            errors.append(f"stn={params[\'stn\']!r} help={params[\'help\']}: {type(exc).__name__}: {exc}")\n    raise RuntimeError("KMA station info returned no AWS stations. Tried variants: " + " | ".join(errors))\n\n\ndef fetch_aws_frame_direct(date_ymd, auth_key):\n    tm = pd.Timestamp(date_ymd).strftime("%Y%m%d") + HOUR_KST\n    aws = fetch_station_info(auth_key, tm)\n    for var in AWSH_COLUMNS:\n        df = fetch_awsh_var(var, tm, auth_key)\n        aws = aws.merge(df[["STN", var]], on="STN", how="left")\n    aws = aws.rename(columns=AWS_VALUE_RENAME)\n    wd = np.deg2rad(aws["WD"].to_numpy(dtype=np.float32))\n    ws = aws["WS"].to_numpy(dtype=np.float32)\n    aws["wind_u"] = -ws * np.sin(wd)\n    aws["wind_v"] = -ws * np.cos(wd)\n    keep = ["STN", "LON", "LAT", "avg_temp", "wind_u", "wind_v", "rain_1h", "humidity"]\n    aws = aws[keep].dropna(subset=["LON", "LAT"])\n    if aws.empty:\n        raise RuntimeError(f"KMA AWS frame has no station coordinates for {tm}")\n    return aws, tm\n\ndef idw_grid(points_xy, values, grid, power=2.0, k=12):\n    pts = np.asarray(points_xy, dtype=np.float32)\n    vals = np.asarray(values, dtype=np.float32)\n    valid = np.isfinite(vals)\n    pts, vals = pts[valid], vals[valid]\n    if len(vals) == 0:\n        return np.full((grid["n"], grid["n"]), np.nan, dtype=np.float32)\n    out = np.empty((grid["n"], grid["n"]), dtype=np.float32)\n    flat_x, flat_y, out_flat = grid["xx"].ravel(), grid["yy"].ravel(), out.ravel()\n    for start in range(0, flat_x.size, 4096):\n        end = min(start + 4096, flat_x.size)\n        dx = flat_x[start:end, None] - pts[None, :, 0]\n        dy = flat_y[start:end, None] - pts[None, :, 1]\n        dist = np.sqrt(dx * dx + dy * dy)\n        kk = min(k, dist.shape[1])\n        idx = np.argpartition(dist, kk - 1, axis=1)[:, :kk]\n        d = np.take_along_axis(dist, idx, axis=1)\n        v = vals[idx]\n        w = 1.0 / np.maximum(d, 1e-6) ** power\n        pred = np.sum(w * v, axis=1) / np.sum(w, axis=1)\n        exact = d < 1e-6\n        if exact.any():\n            rows = np.where(exact.any(axis=1))[0]\n            for rr in rows:\n                pred[rr] = v[rr, np.argmax(exact[rr])]\n        out_flat[start:end] = pred.astype(np.float32)\n    return out\n\ndef aws_patch_direct(date_ymd, grid):\n    aws, tm = fetch_aws_frame_direct(date_ymd, KMA_AUTH_KEY)\n    to_xy = Transformer.from_crs("EPSG:4326", grid["crs"], always_xy=True)\n    xs, ys = to_xy.transform(aws["LON"].to_numpy(), aws["LAT"].to_numpy())\n    pts = np.column_stack([xs, ys])\n    stacks = []\n    for band in AWS_BANDS:\n        stacks.append(idw_grid(pts, aws[band].to_numpy(dtype=np.float32), grid)[None])\n    return np.concatenate(stacks, axis=0).astype(np.float32), aws, tm\n\naws, aws_station_frame, aws_tm = aws_patch_direct(landsat_info["selected_landsat_date"], grid)\nprint("aws:", aws.shape, "tm:", aws_tm, "stations:", len(aws_station_frame))\n\n\nEGIS_WMS_URL = "https://api.mcee.go.kr/geoserver/wms?"\nEGIS_COLOR_MATCH_VERSION = "rgbtol_strict10_supersample4_marg8_announced_v1"\nWMS_SUPERSAMPLE = 4\nLV1_CLASSES = [100, 200, 300, 400, 500, 600, 700]\nURBAN_LV2_CLASSES = [110, 120, 130, 140, 150, 160]\nWATER_LV2_CLASSES = [710, 720]\nOTHER_LV1_CLASSES = [200, 300, 400, 500, 600]\nHYBRID_CLASSES = URBAN_LV2_CLASSES + OTHER_LV1_CLASSES + WATER_LV2_CLASSES\nHYBRID_CLASS_NAMES = {110: "urban_residential", 120: "urban_industrial", 130: "urban_commercial", 140: "urban_cultural_public", 150: "urban_transportation", 160: "urban_mixed", 200: "agriculture", 300: "forest", 400: "grassland", 500: "wetland", 600: "barren", 710: "water_inland", 720: "water_marine"}\nHYBRID_FILL_PRIORITY = {cls: i for i, cls in enumerate([720, 710, 110, 120, 130, 140, 150, 160, 300, 200, 400, 500, 600])}\nLV1_COLORS = {100: (255, 0, 0), 200: (238, 233, 7), 300: (42, 75, 45), 400: (57, 150, 38), 500: (124, 34, 126), 600: (89, 206, 202), 700: (6, 2, 250)}\nLV2_COLORS = {110: (254, 230, 194), 120: (192, 132, 132), 130: (237, 131, 184), 140: (246, 113, 138), 150: (247, 65, 42), 160: (246, 177, 18), 210: (255, 255, 191), 220: (247, 249, 102), 230: (223, 220, 115), 240: (184, 177, 44), 250: (184, 145, 18), 310: (51, 160, 44), 320: (10, 79, 64), 330: (51, 102, 51), 410: (161, 213, 148), 420: (96, 126, 51), 510: (180, 167, 208), 520: (153, 116, 153), 610: (193, 219, 236), 620: (159, 242, 255), 710: (62, 167, 255), 720: (23, 57, 255)}\nLV3_COLORS = {111: (254, 230, 194), 112: (223, 193, 111), 121: (192, 132, 132), 131: (237, 131, 184), 132: (223, 176, 164), 141: (246, 113, 138), 151: (229, 38, 254), 152: (197, 50, 81), 153: (252, 4, 78), 154: (247, 65, 42), 155: (115, 0, 0), 161: (246, 177, 18), 162: (255, 122, 0), 163: (199, 88, 27), 211: (255, 255, 191), 212: (244, 230, 168), 221: (247, 249, 102), 222: (245, 228, 10), 231: (223, 220, 115), 241: (184, 177, 44), 251: (184, 145, 18), 252: (170, 100, 0), 311: (51, 160, 44), 321: (10, 79, 64), 331: (51, 102, 51), 411: (161, 213, 148), 421: (128, 228, 90), 422: (113, 176, 90), 423: (96, 126, 51), 511: (180, 167, 208), 521: (153, 116, 153), 522: (124, 30, 162), 611: (193, 219, 236), 612: (171, 197, 202), 613: (171, 182, 165), 621: (88, 90, 138), 622: (123, 181, 172), 623: (159, 242, 255), 711: (62, 167, 255), 712: (93, 109, 255), 721: (23, 57, 255)}\nCOLOR_TABLE_BY_LEVEL = {"lv1": LV1_COLORS, "lv2": LV2_COLORS, "lv3": LV3_COLORS}\nWMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL = {"lv1": 18.0, "lv2": 18.0, "lv3": 18.0}\nWMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL = {"lv1": 10.0, "lv2": 10.0, "lv3": 10.0}\nWMS_COLOR_MARGIN_MIN_BY_LEVEL = {"lv1": 8.0, "lv2": 8.0, "lv3": 8.0}\nFILL_CLASS_BY_CHOICE = {"sea": 720, "urban": 110}\nEGIS_REFERENCE_FILL_MAX_YEAR = 2021\nEGIS_REFERENCE_YEAR = 2022\nEGIS_REFERENCE_FILL_WEIGHT = 4\nEGIS_MAX_FILL_RADIUS_PX = 16\nEGIS_PIXEL_SIZE_M = 60.0\n\ndef egis_policy_for_year(year):\n    year = int(year)\n    if year <= 2015:\n        return {"policy_year": 2007, "source_key": "lv2_2007_g", "source_level": "lv2", "layer": "EGIS:lv2_2007_g"}\n    rows = [\n        (2016, 2018, 2018, "lv3_10th_g", "lv3"),\n        (2019, 2019, 2019, "lv3_2020_g", "lv3"),\n        (2020, 2020, 2020, "lv3_2021_g", "lv3"),\n        (2021, 2021, 2021, "lv2_2022y", "lv2"),\n        (2022, 2022, 2022, "lv2_2023y", "lv2"),\n        (2023, 2023, 2023, "lv2_2024y", "lv2"),\n        (2024, 2026, 2024, "lv2_2025y", "lv2"),\n    ]\n    for amin, amax, py, key, level in rows:\n        if amin <= year <= amax:\n            return {"policy_year": py, "source_key": key, "source_level": level, "layer": f"EGIS:{key}"}\n    raise ValueError(f"unsupported EGIS analysis year: {year}")\n\ndef code_to_lv1(code):\n    code = int(code)\n    if code <= 0:\n        return 0\n    if code in LV1_CLASSES:\n        return code\n    return (code // 100) * 100\n\ndef code_to_lv2(code, level="lv3"):\n    code = int(code)\n    if code <= 0:\n        return 0\n    if level in {"lv1", "lv2"}:\n        return code\n    return (code // 10) * 10\n\ndef code_to_hybrid(code, level="lv3"):\n    code = int(code)\n    if code <= 0:\n        return 0\n    lv1 = code_to_lv1(code)\n    lv2 = code_to_lv2(code, level)\n    if lv1 == 100:\n        return lv2 if lv2 in URBAN_LV2_CLASSES else 110\n    if lv1 == 700:\n        return lv2 if lv2 in WATER_LV2_CLASSES else 720\n    if lv1 in OTHER_LV1_CLASSES:\n        return lv1\n    return 0\n\ndef request_wms_image(layer_name, grid, scale=1):\n    width = int(grid["n"] * scale)\n    height = int(grid["n"] * scale)\n    params = {"SERVICE": "WMS", "VERSION": "1.1.1", "REQUEST": "GetMap", "LAYERS": layer_name, "STYLES": "", "SRS": grid["crs"], "BBOX": f\'{grid["left"]},{grid["bottom"]},{grid["right"]},{grid["top"]}\', "WIDTH": width, "HEIGHT": height, "FORMAT": "image/png", "TRANSPARENT": "TRUE"}\n    last_exc = None\n    for attempt in range(1, 6):\n        try:\n            r = requests.get(EGIS_WMS_URL, params=params, timeout=REQUEST_TIMEOUT)\n            content_type = r.headers.get("content-type", "")\n            if not r.ok:\n                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")\n            if "image" not in content_type.lower() and not r.content.startswith(bytes([137, 80, 78, 71])):\n                raise RuntimeError(f"WMS did not return image: {content_type}; {r.text[:500]}")\n            img = Image.open(BytesIO(r.content)).convert("RGBA")\n            if img.size != (width, height):\n                img = img.resize((width, height), Image.Resampling.NEAREST)\n            return np.array(img)\n        except Exception as exc:\n            last_exc = exc\n            wait = min(5 * attempt, 30)\n            print(f"[EGIS WMS retry {attempt}/5] {layer_name}: {type(exc).__name__}: {exc}; wait {wait}s")\n            time.sleep(wait)\n    raise RuntimeError(f"EGIS WMS failed for {layer_name}: {last_exc}")\n\ndef rgb_to_hybrid_landcover(rgba, level):\n    rgb = rgba[..., :3].astype(np.float32).reshape(-1, 3)\n    alpha = rgba[..., 3].reshape(-1)\n    table = COLOR_TABLE_BY_LEVEL[level]\n    codes = np.array(list(table.keys()), dtype=np.int16)\n    colors = np.array([table[int(c)] for c in codes], dtype=np.float32)\n    raw_flat = np.zeros(rgb.shape[0], dtype=np.int16)\n    accepted_flat = np.zeros(rgb.shape[0], dtype=np.uint8)\n    dist_flat = np.full(rgb.shape[0], np.inf, dtype=np.float32)\n    absmax_flat = np.full(rgb.shape[0], np.inf, dtype=np.float32)\n    dist_thr = WMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL[level]\n    ch_thr = WMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL[level]\n    margin_thr = WMS_COLOR_MARGIN_MIN_BY_LEVEL[level]\n    for start in range(0, rgb.shape[0], 200_000):\n        end = min(start + 200_000, rgb.shape[0])\n        chunk = rgb[start:end]\n        diff = chunk[:, None, :] - colors[None, :, :]\n        dist_sq = np.sum(diff * diff, axis=-1)\n        best_idx = np.argmin(dist_sq, axis=-1)\n        best_dist = np.sqrt(dist_sq[np.arange(end - start), best_idx])\n        second_sq = np.partition(dist_sq, 1, axis=-1)[:, 1] if dist_sq.shape[1] > 1 else np.full(end - start, np.inf)\n        margin = np.sqrt(second_sq) - best_dist\n        best_absmax = np.max(np.abs(diff[np.arange(end - start), best_idx, :]), axis=-1)\n        accepted = (alpha[start:end] >= 128) & (best_dist <= dist_thr) & (best_absmax <= ch_thr) & (margin >= margin_thr)\n        raw_flat[start:end] = codes[best_idx]\n        accepted_flat[start:end] = accepted.astype(np.uint8)\n        dist_flat[start:end] = best_dist.astype(np.float32)\n        absmax_flat[start:end] = best_absmax.astype(np.float32)\n    hybrid_flat = np.zeros_like(raw_flat)\n    raw_to_hybrid = {int(code): int(code_to_hybrid(int(code), level)) for code in codes}\n    idx = np.where(accepted_flat.astype(bool))[0]\n    hybrid_flat[idx] = np.array([raw_to_hybrid[int(c)] for c in raw_flat[idx]], dtype=np.int16)\n    shape = rgba.shape[:2]\n    source_nodata = (rgba[..., 3] < 128)\n    unrecognized = (~source_nodata) & (accepted_flat.reshape(shape) == 0)\n    return hybrid_flat.reshape(shape), raw_flat.reshape(shape), source_nodata.astype(np.uint8), unrecognized.astype(np.uint8), dist_flat.reshape(shape), absmax_flat.reshape(shape)\n\ndef downsample_rgba_preview(rgba, out_n):\n    return np.array(Image.fromarray(rgba.astype(np.uint8), mode="RGBA").resize((out_n, out_n), Image.Resampling.NEAREST), dtype=np.uint8)\n\ndef block_majority_bool(arr, factor):\n    h, w = arr.shape\n    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)\n    return (blocks.mean(axis=-1) >= 0.5).astype(np.uint8)\n\ndef block_mean_float(arr, factor):\n    h, w = arr.shape\n    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)\n    return np.nanmean(blocks.astype(np.float32), axis=-1).astype(np.float32)\n\ndef block_max_float(arr, factor):\n    h, w = arr.shape\n    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)\n    return np.nanmax(blocks.astype(np.float32), axis=-1).astype(np.float32)\n\ndef block_mode_codes(arr, factor, classes, priority=None):\n    h, w = arr.shape\n    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)\n    out = np.zeros(blocks.shape[:2], dtype=arr.dtype)\n    best_count = np.zeros(blocks.shape[:2], dtype=np.int16)\n    ordered = sorted([int(c) for c in classes], key=lambda c: (priority or {}).get(c, 9999))\n    for cls in ordered:\n        counts = (blocks == int(cls)).sum(axis=-1).astype(np.int16)\n        take = counts > best_count\n        out[take] = int(cls)\n        best_count[take] = counts[take]\n    return out\n\ndef egis_direct_for_policy(policy, grid):\n    rgba_hr = request_wms_image(policy["layer"], grid, scale=WMS_SUPERSAMPLE)\n    hybrid_hr, raw_code_hr, source_nodata_hr, unrecognized_hr, dist_hr, absmax_hr = rgb_to_hybrid_landcover(rgba_hr, policy["source_level"])\n    priority = {0: 9999, **HYBRID_FILL_PRIORITY}\n    raw_classes = [0] + [int(c) for c in COLOR_TABLE_BY_LEVEL[policy["source_level"]].keys()]\n    major = block_mode_codes(hybrid_hr.astype(np.int16), WMS_SUPERSAMPLE, [0] + HYBRID_CLASSES, priority)\n    raw_code = block_mode_codes(raw_code_hr.astype(np.int16), WMS_SUPERSAMPLE, raw_classes, {0: 9999})\n    source_nodata = block_majority_bool(source_nodata_hr, WMS_SUPERSAMPLE)\n    unrecognized = block_majority_bool(unrecognized_hr, WMS_SUPERSAMPLE)\n    rgba = downsample_rgba_preview(rgba_hr, grid["n"])\n    color_diag = {\n        "egis_color_match_version": np.array(EGIS_COLOR_MATCH_VERSION),\n        "egis_wms_supersample": np.array(int(WMS_SUPERSAMPLE)),\n        "egis_rgb_distance_threshold": np.array(float(WMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL[policy["source_level"]]), dtype=np.float32),\n        "egis_rgb_channel_tolerance": np.array(float(WMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL[policy["source_level"]]), dtype=np.float32),\n        "egis_rgb_margin_min": np.array(float(WMS_COLOR_MARGIN_MIN_BY_LEVEL[policy["source_level"]]), dtype=np.float32),\n        "egis_rgb_distance_mean_60m": block_mean_float(dist_hr, WMS_SUPERSAMPLE),\n        "egis_rgb_distance_max_60m": block_max_float(dist_hr, WMS_SUPERSAMPLE),\n        "egis_rgb_absmax_mean_60m": block_mean_float(absmax_hr, WMS_SUPERSAMPLE),\n        "egis_rgb_absmax_max_60m": block_max_float(absmax_hr, WMS_SUPERSAMPLE),\n        "egis_unrecognized_fraction_60m": block_mean_float(unrecognized_hr.astype(np.float32), WMS_SUPERSAMPLE),\n        "egis_source_nodata_fraction_60m": block_mean_float(source_nodata_hr.astype(np.float32), WMS_SUPERSAMPLE),\n    }\n    major[source_nodata.astype(bool) | unrecognized.astype(bool)] = 0\n    return policy, rgba, major.astype(np.int16), raw_code.astype(np.int16), source_nodata, unrecognized, color_diag\n\ndef egis_direct(row, grid):\n    policy = egis_policy_for_year(pd.Timestamp(row["date"]).year)\n    return egis_direct_for_policy(policy, grid)\n\negis_policy, egis_rgba, egis_major, egis_raw_code, egis_source_nodata, egis_unrecognized, egis_color_diag = egis_direct(row, grid)\nraw_egis_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_{egis_policy[\'source_key\']}.npz"\nnp.savez_compressed(raw_egis_npz, raw_rgba=egis_rgba, egis_hybrid_lc_unfilled_60m=egis_major, egis_raw_code_60m=egis_raw_code, egis_source_nodata_mask=egis_source_nodata, egis_unrecognized_rgb_mask=egis_unrecognized, **egis_color_diag, **{k: np.array(v) for k, v in egis_policy.items()})\nprint("EGIS direct:", egis_policy, "raw:", raw_egis_npz)\n\n\ndef hybrid_to_rgba(arr):\n    out = np.zeros((*arr.shape, 4), dtype=np.uint8)\n    colors = {**LV1_COLORS, **LV2_COLORS}\n    for cls in HYBRID_CLASSES:\n        m = arr == int(cls)\n        out[m, :3] = np.array(colors.get(int(cls), (0, 0, 0)), dtype=np.uint8)\n        out[m, 3] = 255\n    out[arr == 0] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    return out\n\ndef mask_to_rgba(mask, on=(255, 0, 255), off=(0, 0, 0)):\n    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)\n    rgb[~mask.astype(bool)] = np.array(off, dtype=np.uint8)\n    rgb[mask.astype(bool)] = np.array(on, dtype=np.uint8)\n    return np.dstack([rgb, np.full(mask.shape, 255, dtype=np.uint8)])\n\ndef add_label(img, label, height=24):\n    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))\n    base.paste(img, (0, height))\n    draw = ImageDraw.Draw(base)\n    draw.text((4, 5), str(label)[:80], fill=(0, 0, 0, 255))\n    return base\n\ndef save_sheet(items, out_path):\n    labeled = [add_label(Image.fromarray(arr).convert("RGBA").resize((256, 256), Image.Resampling.NEAREST), title) for title, arr in items]\n    cols = 3\n    rows = int(math.ceil(len(labeled) / cols))\n    sheet = Image.new("RGBA", (cols * 256, rows * 280), (255, 255, 255, 255))\n    for i, img in enumerate(labeled):\n        sheet.paste(img, ((i % cols) * 256, (i // cols) * 280))\n    sheet.save(out_path)\n\ndef choose_weighted_majority(counts):\n    best_cls = 0\n    best_count = 0\n    best_priority = 9999\n    for cls in HYBRID_CLASSES:\n        count = int(counts[int(cls)]) if int(cls) < len(counts) else 0\n        priority = HYBRID_FILL_PRIORITY.get(int(cls), 9999)\n        if count > best_count or (count == best_count and count > 0 and priority < best_priority):\n            best_cls = int(cls)\n            best_count = count\n            best_priority = priority\n    return best_cls\n\ndef neighbor_counts(filled, y, x):\n    y0, y1 = max(0, y - 1), min(filled.shape[0], y + 2)\n    x0, x1 = max(0, x - 1), min(filled.shape[1], x + 2)\n    vals = filled[y0:y1, x0:x1].reshape(-1)\n    vals = vals[vals > 0]\n    counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)\n    if vals.size:\n        uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)\n        counts[uniq] += cnt.astype(np.int16)\n    return counts\n\ndef fill_by_iterative_neighbors(major, target_mask, reference_lc=None, reference_weight=0, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):\n    filled = major.copy().astype(np.int16)\n    pending = target_mask.astype(bool).copy()\n    fill_mask = np.zeros_like(pending, dtype=bool)\n    reference_vote_mask = np.zeros_like(pending, dtype=bool)\n    radius_used = np.zeros(filled.shape, dtype=np.int16)\n    for radius in range(1, int(max_radius_px) + 1):\n        ys, xs = np.where(pending)\n        if len(ys) == 0:\n            break\n        updates = []\n        used_reference = []\n        for y, x in zip(ys, xs):\n            counts = neighbor_counts(filled, int(y), int(x))\n            ref_used = False\n            if reference_lc is not None:\n                ref_cls = int(reference_lc[int(y), int(x)])\n                if ref_cls > 0:\n                    counts[ref_cls] += int(reference_weight)\n                    ref_used = True\n            winner = choose_weighted_majority(counts)\n            if winner > 0:\n                updates.append((int(y), int(x), int(winner)))\n                used_reference.append(ref_used)\n        if not updates:\n            break\n        for (y, x, winner), ref_used in zip(updates, used_reference):\n            filled[y, x] = winner\n            pending[y, x] = False\n            fill_mask[y, x] = True\n            reference_vote_mask[y, x] = bool(ref_used)\n            radius_used[y, x] = radius\n    return filled, fill_mask, reference_vote_mask, pending, radius_used\n\ndef fill_by_expanding_window(major, target_mask, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):\n    filled = major.copy().astype(np.int16)\n    pending = target_mask.astype(bool).copy()\n    fill_mask = np.zeros_like(pending, dtype=bool)\n    radius_used = np.zeros(filled.shape, dtype=np.int16)\n    for radius in range(1, int(max_radius_px) + 1):\n        ys, xs = np.where(pending)\n        if len(ys) == 0:\n            break\n        updates = []\n        for y, x in zip(ys, xs):\n            y0, y1 = max(0, int(y) - radius), min(filled.shape[0], int(y) + radius + 1)\n            x0, x1 = max(0, int(x) - radius), min(filled.shape[1], int(x) + radius + 1)\n            vals = filled[y0:y1, x0:x1].reshape(-1)\n            vals = vals[vals > 0]\n            if vals.size == 0:\n                continue\n            counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)\n            uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)\n            counts[uniq] += cnt.astype(np.int16)\n            winner = choose_weighted_majority(counts)\n            if winner > 0:\n                updates.append((int(y), int(x), int(winner)))\n        if not updates:\n            continue\n        for y, x, winner in updates:\n            filled[y, x] = winner\n            pending[y, x] = False\n            fill_mask[y, x] = True\n            radius_used[y, x] = radius\n    return filled, fill_mask, pending, radius_used\n\ndef load_reference_egis_lc_for_fill():\n    ref_policy = egis_policy_for_year(EGIS_REFERENCE_YEAR)\n    ref_policy, ref_rgba, ref_major, ref_raw_code, ref_source_nodata, ref_unrecognized, ref_color_diag = egis_direct_for_policy(ref_policy, grid)\n    ref_unknown = (ref_major == 0) | ref_source_nodata.astype(bool) | ref_unrecognized.astype(bool)\n    if bool(ref_unknown.any()):\n        ref_major, ref_majority_mask, ref_leftover, ref_radius = fill_by_expanding_window(ref_major, ref_unknown, EGIS_MAX_FILL_RADIUS_PX)\n        if bool(ref_leftover.any()):\n            print(f"2022 reference still has {int(ref_leftover.sum())} unknown pixels after radius cap; ignored as reference votes.")\n            ref_major[ref_leftover] = 0\n    ref_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_reference_{ref_policy[\'source_key\']}.npz"\n    np.savez_compressed(\n        ref_npz,\n        raw_rgba=ref_rgba,\n        egis_hybrid_lc_ref_filled_60m=ref_major.astype(np.int16),\n        egis_raw_code_60m=ref_raw_code.astype(np.int16),\n        egis_source_nodata_mask=ref_source_nodata.astype(np.uint8),\n        egis_unrecognized_rgb_mask=ref_unrecognized.astype(np.uint8),\n        reference_year=np.array(int(EGIS_REFERENCE_YEAR)),\n        reference_fill_radius_cap_px=np.array(int(EGIS_MAX_FILL_RADIUS_PX)),\n        **{f"reference_{k}": v for k, v in ref_color_diag.items()},\n        **{k: np.array(v) for k, v in ref_policy.items()},\n    )\n    return ref_major.astype(np.int16), ref_npz\n\ndef auto_fill_mask(major, target_mask, analysis_year, stats_prefix):\n    stats = {\n        f"{stats_prefix}_fill_method": "none",\n        f"{stats_prefix}_input_pixels": int(target_mask.sum()),\n        f"{stats_prefix}_reference_fill_year": -1,\n        f"{stats_prefix}_reference_fill_weight": 0,\n        f"{stats_prefix}_reference_vote_pixels": 0,\n        f"{stats_prefix}_majority_fill_pixels": 0,\n        f"{stats_prefix}_fill_max_radius_px": 0,\n        f"{stats_prefix}_fill_max_radius_m": 0.0,\n        f"{stats_prefix}_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),\n        f"{stats_prefix}_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),\n        f"{stats_prefix}_leftover_pixels": 0,\n        f"{stats_prefix}_reference_raw_npz": "",\n    }\n    if not bool(target_mask.any()):\n        return major.copy().astype(np.int16), stats\n    reference_lc = None\n    reference_weight = 0\n    if int(analysis_year) <= EGIS_REFERENCE_FILL_MAX_YEAR:\n        reference_lc, ref_npz = load_reference_egis_lc_for_fill()\n        reference_weight = int(EGIS_REFERENCE_FILL_WEIGHT)\n        stats.update({\n            f"{stats_prefix}_fill_method": "attempt3_8nbr_plus_2022x4",\n            f"{stats_prefix}_reference_fill_year": int(EGIS_REFERENCE_YEAR),\n            f"{stats_prefix}_reference_fill_weight": int(EGIS_REFERENCE_FILL_WEIGHT),\n            f"{stats_prefix}_reference_raw_npz": str(ref_npz),\n        })\n    else:\n        stats[f"{stats_prefix}_fill_method"] = "attempt3_expanding_neighbor_majority"\n    filled, neighbor_mask, ref_vote_mask, pending, radius_used = fill_by_iterative_neighbors(\n        major,\n        target_mask,\n        reference_lc=reference_lc,\n        reference_weight=reference_weight,\n        max_radius_px=EGIS_MAX_FILL_RADIUS_PX,\n    )\n    if bool(pending.any()):\n        filled, window_mask, pending, window_radius = fill_by_expanding_window(filled, pending, EGIS_MAX_FILL_RADIUS_PX)\n        radius_used = np.maximum(radius_used, window_radius)\n        neighbor_mask = neighbor_mask | window_mask\n        stats[f"{stats_prefix}_fill_method"] += "_plus_bounded_window_majority"\n    stats[f"{stats_prefix}_reference_vote_pixels"] = int(ref_vote_mask.sum())\n    stats[f"{stats_prefix}_majority_fill_pixels"] = int(neighbor_mask.sum())\n    stats[f"{stats_prefix}_leftover_pixels"] = int(pending.sum())\n    stats[f"{stats_prefix}_fill_max_radius_px"] = int(radius_used.max()) if radius_used.size else 0\n    stats[f"{stats_prefix}_fill_max_radius_m"] = float(stats[f"{stats_prefix}_fill_max_radius_px"] * EGIS_PIXEL_SIZE_M)\n    if bool(pending.any()):\n        raise RuntimeError(\n            f"EGIS {stats_prefix} fill left {int(pending.sum())} pixels after {EGIS_MAX_FILL_RADIUS_PX}px "\n            f"({EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M:.0f}m) cap."\n        )\n    return filled.astype(np.int16), stats\n\n# Step 1: resolve unrecognized RGB first. User choice is applied only after this step.\negis_major_raw = egis_major.copy().astype(np.int16)\negis_unrecognized_mask = egis_unrecognized.astype(bool) & (egis_major_raw == 0)\negis_major, egis_unrecognized_fill_stats = auto_fill_mask(\n    egis_major_raw,\n    egis_unrecognized_mask,\n    analysis_year,\n    "egis_unrecognized",\n)\nunrecognized_filled_npz = DIRECT_RAW_DIR / f"egis_unrecognized_filled_{RUN_LABEL}_{egis_policy[\'source_key\']}.npz"\nnp.savez_compressed(\n    unrecognized_filled_npz,\n    egis_hybrid_lc_before_unrecognized_fill_60m=egis_major_raw.astype(np.int16),\n    egis_hybrid_lc_after_unrecognized_fill_60m=egis_major.astype(np.int16),\n    egis_unrecognized_rgb_mask=egis_unrecognized.astype(np.uint8),\n    egis_source_nodata_mask=egis_source_nodata.astype(np.uint8),\n    **{k: np.array(v) for k, v in egis_unrecognized_fill_stats.items()},\n)\negis_unrecognized_fill_stats["egis_unrecognized_filled_npz"] = str(unrecognized_filled_npz)\nprint("EGIS unrecognized fill stats:", egis_unrecognized_fill_stats)\n\ndef save_unknown_review():\n    # attempt3 policy: show remaining source nodata/zero pixels, then fill them automatically by majority/reference.\n    unknown = ((egis_source_nodata.astype(bool) | (egis_major == 0)) & (egis_major == 0))\n    overlay = egis_rgba[..., :3].copy()\n    overlay[unknown] = np.array([255, 0, 255], dtype=np.uint8)\n    overlay = np.dstack([overlay, np.full(unknown.shape, 255, dtype=np.uint8)])\n    png = REVIEW_DIR / f"unknown_{RUN_LABEL}_{egis_policy[\'source_key\']}.png"\n    save_sheet([\n        ("raw EGIS RGBA", egis_rgba),\n        ("hybrid before unrecognized fill", hybrid_to_rgba(egis_major_raw)),\n        (f"unrecognized filled={int(egis_unrecognized_mask.sum())}", mask_to_rgba(egis_unrecognized_mask, on=(255, 128, 0))),\n        ("hybrid after unrecognized fill", hybrid_to_rgba(egis_major)),\n        (f"unknown for auto fill={int(unknown.sum())}", mask_to_rgba(unknown)),\n        (f"source_nodata={int(egis_source_nodata.sum())}", mask_to_rgba(egis_source_nodata.astype(bool), on=(255, 255, 255))),\n    ], png)\n    row_out = {\n        "run_label": RUN_LABEL,\n        "roi_label": ROI_LABEL,\n        "requested_date": landsat_info["requested_date"],\n        "selected_landsat_date": landsat_info["selected_landsat_date"],\n        "roi_center_lat": float(ROI_CENTER_LAT),\n        "roi_center_lon": float(ROI_CENTER_LON),\n        "policy_year": int(egis_policy["policy_year"]),\n        "source_key": egis_policy["source_key"],\n        "source_level": egis_policy["source_level"],\n        "raw_direct_egis_npz": str(raw_egis_npz),\n        "unrecognized_filled_npz": str(unrecognized_filled_npz),\n        "unrecognized_pixels": int(egis_unrecognized_mask.sum()),\n        "unknown_pixels": int(unknown.sum()),\n        "unknown_ratio": float(unknown.mean()),\n        "preview_png": str(png),\n        "fill_choice": "auto_attempt3",\n        "fill_class": -1,\n    }\n    row_out.update(egis_unrecognized_fill_stats)\n    pd.DataFrame([row_out]).to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")\n    pd.DataFrame([row_out]).to_csv(DECISION_CSV, index=False, encoding="utf-8-sig")\n    return unknown, png\n\nunknown_mask, preview_png = save_unknown_review()\nprint("unknown review png:", preview_png)\nprint("unknown pixels for automatic fill:", int(unknown_mask.sum()), "ratio:", float(unknown_mask.mean()))\nprint("decision csv:", DECISION_CSV)\n\n\ndef read_fill_choice():\n    # attempt3와 동일하게 바다/도심 수동 강제 없이 남은 source_nodata/zero만 자동 보간한다.\n    if not bool(unknown_mask.any()):\n        return "none", None\n    print("EGIS unknown fill selected: auto_attempt3 (no sea/urban forced fill)")\n    return "auto", None\n\ndef record_fill_decision(choice, cls, stats):\n    if not DECISION_CSV.exists():\n        return\n    df = pd.read_csv(DECISION_CSV)\n    if df.empty:\n        return\n    df.loc[0, "fill_choice"] = "auto_attempt3" if choice == "auto" else choice\n    df.loc[0, "fill_class"] = -1\n    for key, value in stats.items():\n        if isinstance(value, (np.integer, np.floating)):\n            value = value.item()\n        df.loc[0, key] = value\n    df.to_csv(DECISION_CSV, index=False, encoding="utf-8-sig")\n\ndef egis_onehot_from_lc(lc):\n    if (lc == 0).any():\n        raise RuntimeError("EGIS landcover still has unknown pixels")\n    arrays, names = [], []\n    for cls in HYBRID_CLASSES:\n        label = HYBRID_CLASS_NAMES[int(cls)]\n        arrays.append((lc == int(cls)).astype(np.float32)[None])\n        names.append(f"egis_cat_{int(cls)}_{label}")\n    return np.concatenate(arrays, axis=0), names\n\ndef sample_meta(date_ymd, sun_azimuth, sun_elevation, lat, lon):\n    doy = pd.Timestamp(date_ymd).dayofyear\n    return np.array([\n        math.sin(2 * math.pi * doy / 366.0),\n        math.cos(2 * math.pi * doy / 366.0),\n        float(sun_azimuth) / 360.0,\n        float(sun_elevation) / 90.0,\n        (float(lat) - LAT_META_MEAN) / LAT_META_STD,\n        (float(lon) - LON_META_MEAN) / LON_META_STD,\n    ], dtype=np.float32)\n\ndef choose_weighted_majority(counts):\n    best_cls = 0\n    best_count = 0\n    best_priority = 9999\n    for cls in HYBRID_CLASSES:\n        count = int(counts[int(cls)]) if int(cls) < len(counts) else 0\n        priority = HYBRID_FILL_PRIORITY.get(int(cls), 9999)\n        if count > best_count or (count == best_count and count > 0 and priority < best_priority):\n            best_cls = int(cls)\n            best_count = count\n            best_priority = priority\n    return best_cls\n\ndef neighbor_counts(filled, y, x):\n    y0, y1 = max(0, y - 1), min(filled.shape[0], y + 2)\n    x0, x1 = max(0, x - 1), min(filled.shape[1], x + 2)\n    vals = filled[y0:y1, x0:x1].reshape(-1)\n    vals = vals[vals > 0]\n    counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)\n    if vals.size:\n        uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)\n        counts[uniq] += cnt.astype(np.int16)\n    return counts\n\ndef fill_by_iterative_neighbors(major, target_mask, reference_lc=None, reference_weight=0, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):\n    filled = major.copy().astype(np.int16)\n    pending = target_mask.astype(bool).copy()\n    fill_mask = np.zeros_like(pending, dtype=bool)\n    reference_vote_mask = np.zeros_like(pending, dtype=bool)\n    radius_used = np.zeros(filled.shape, dtype=np.int16)\n    for radius in range(1, int(max_radius_px) + 1):\n        ys, xs = np.where(pending)\n        if len(ys) == 0:\n            break\n        updates = []\n        used_reference = []\n        for y, x in zip(ys, xs):\n            counts = neighbor_counts(filled, int(y), int(x))\n            ref_used = False\n            if reference_lc is not None:\n                ref_cls = int(reference_lc[int(y), int(x)])\n                if ref_cls > 0:\n                    counts[ref_cls] += int(reference_weight)\n                    ref_used = True\n            winner = choose_weighted_majority(counts)\n            if winner > 0:\n                updates.append((int(y), int(x), int(winner)))\n                used_reference.append(ref_used)\n        if not updates:\n            break\n        for (y, x, winner), ref_used in zip(updates, used_reference):\n            filled[y, x] = winner\n            pending[y, x] = False\n            fill_mask[y, x] = True\n            reference_vote_mask[y, x] = bool(ref_used)\n            radius_used[y, x] = radius\n    return filled, fill_mask, reference_vote_mask, pending, radius_used\n\ndef fill_by_expanding_window(major, target_mask, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):\n    filled = major.copy().astype(np.int16)\n    pending = target_mask.astype(bool).copy()\n    fill_mask = np.zeros_like(pending, dtype=bool)\n    radius_used = np.zeros(filled.shape, dtype=np.int16)\n    for radius in range(1, int(max_radius_px) + 1):\n        ys, xs = np.where(pending)\n        if len(ys) == 0:\n            break\n        updates = []\n        for y, x in zip(ys, xs):\n            y0, y1 = max(0, int(y) - radius), min(filled.shape[0], int(y) + radius + 1)\n            x0, x1 = max(0, int(x) - radius), min(filled.shape[1], int(x) + radius + 1)\n            vals = filled[y0:y1, x0:x1].reshape(-1)\n            vals = vals[vals > 0]\n            if vals.size == 0:\n                continue\n            counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)\n            uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)\n            counts[uniq] += cnt.astype(np.int16)\n            winner = choose_weighted_majority(counts)\n            if winner > 0:\n                updates.append((int(y), int(x), int(winner)))\n        if not updates:\n            continue\n        for y, x, winner in updates:\n            filled[y, x] = winner\n            pending[y, x] = False\n            fill_mask[y, x] = True\n            radius_used[y, x] = radius\n    return filled, fill_mask, pending, radius_used\n\ndef auto_fill_egis_unknown(major, target_mask, analysis_year):\n    year = int(analysis_year)\n    stats = {\n        "egis_fill_method": "majority",\n        "egis_reference_fill_year": -1,\n        "egis_reference_fill_weight": 0,\n        "egis_reference_vote_pixels": 0,\n        "egis_majority_fill_pixels": 0,\n        "egis_manual_fill_pixels": 0,\n        "egis_fill_max_radius_px": 0,\n        "egis_fill_max_radius_m": 0.0,\n        "egis_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),\n        "egis_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),\n        "egis_unknown_leftover_pixels": 0,\n        "egis_reference_raw_npz": "",\n    }\n    if not bool(target_mask.any()):\n        stats["egis_fill_method"] = "none"\n        return major.copy().astype(np.int16), stats\n    reference_lc = None\n    if year <= EGIS_REFERENCE_FILL_MAX_YEAR:\n        ref_policy = egis_policy_for_year(EGIS_REFERENCE_YEAR)\n        ref_policy, ref_rgba, ref_major, ref_raw_code, ref_source_nodata, ref_unrecognized, ref_color_diag = egis_direct_for_policy(ref_policy, grid)\n        ref_unknown = (ref_major == 0) | ref_source_nodata.astype(bool) | ref_unrecognized.astype(bool)\n        if bool(ref_unknown.any()):\n            ref_major, ref_majority_mask, ref_leftover, ref_radius = fill_by_expanding_window(ref_major, ref_unknown, EGIS_MAX_FILL_RADIUS_PX)\n            if bool(ref_leftover.any()):\n                print(f"2022 reference still has {int(ref_leftover.sum())} unknown pixels after radius cap; those pixels are ignored as reference votes.")\n                ref_major[ref_leftover] = 0\n        ref_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_reference_{ref_policy[\'source_key\']}.npz"\n        np.savez_compressed(\n            ref_npz,\n            raw_rgba=ref_rgba,\n            egis_hybrid_lc_ref_filled_60m=ref_major.astype(np.int16),\n            egis_raw_code_60m=ref_raw_code.astype(np.int16),\n            egis_source_nodata_mask=ref_source_nodata.astype(np.uint8),\n            egis_unrecognized_rgb_mask=ref_unrecognized.astype(np.uint8),\n            reference_year=np.array(int(EGIS_REFERENCE_YEAR)),\n            reference_fill_radius_cap_px=np.array(int(EGIS_MAX_FILL_RADIUS_PX)),\n            **{f"reference_{k}": v for k, v in ref_color_diag.items()},\n            **{k: np.array(v) for k, v in ref_policy.items()},\n        )\n        reference_lc = ref_major.astype(np.int16)\n        stats.update({\n            "egis_fill_method": "attempt3_8nbr_plus_2022x4",\n            "egis_reference_fill_year": int(EGIS_REFERENCE_YEAR),\n            "egis_reference_fill_weight": int(EGIS_REFERENCE_FILL_WEIGHT),\n            "egis_reference_raw_npz": str(ref_npz),\n        })\n    filled, neighbor_mask, ref_vote_mask, pending, radius_used = fill_by_iterative_neighbors(\n        major,\n        target_mask,\n        reference_lc=reference_lc,\n        reference_weight=EGIS_REFERENCE_FILL_WEIGHT if reference_lc is not None else 0,\n        max_radius_px=EGIS_MAX_FILL_RADIUS_PX,\n    )\n    if bool(pending.any()):\n        # Final bounded window majority for islands that had no 8-neighbor propagation path.\n        filled, window_mask, pending, window_radius = fill_by_expanding_window(filled, pending, EGIS_MAX_FILL_RADIUS_PX)\n        radius_used = np.maximum(radius_used, window_radius)\n        neighbor_mask = neighbor_mask | window_mask\n        stats["egis_fill_method"] += "_plus_bounded_window_majority"\n    stats["egis_reference_vote_pixels"] = int(ref_vote_mask.sum())\n    stats["egis_majority_fill_pixels"] = int(neighbor_mask.sum())\n    stats["egis_unknown_leftover_pixels"] = int(pending.sum())\n    stats["egis_fill_max_radius_px"] = int(radius_used.max()) if radius_used.size else 0\n    stats["egis_fill_max_radius_m"] = float(stats["egis_fill_max_radius_px"] * EGIS_PIXEL_SIZE_M)\n    if bool(pending.any()):\n        raise RuntimeError(\n            f"EGIS auto fill left {int(pending.sum())} unknown pixels after {EGIS_MAX_FILL_RADIUS_PX}px "\n            f"({EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M:.0f}m) cap. Rerun with manual sea/urban or increase EGIS_MAX_FILL_RADIUS_PX."\n        )\n    return filled.astype(np.int16), stats\n\ndef apply_egis_fill(major, target_mask, choice, fill_class):\n    if choice == "none":\n        stats = {\n            "egis_fill_method": "none",\n            "egis_reference_fill_year": -1,\n            "egis_reference_fill_weight": 0,\n            "egis_reference_vote_pixels": 0,\n            "egis_majority_fill_pixels": 0,\n            "egis_manual_fill_pixels": 0,\n            "egis_fill_max_radius_px": 0,\n            "egis_fill_max_radius_m": 0.0,\n            "egis_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),\n            "egis_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),\n            "egis_unknown_leftover_pixels": 0,\n            "egis_reference_raw_npz": "",\n        }\n        return major.copy().astype(np.int16), stats\n    if choice == "auto":\n        return auto_fill_egis_unknown(major, target_mask, analysis_year)\n    raise RuntimeError("EGIS fill은 attempt3 자동 보간만 허용합니다. 바다/도심 강제 채움은 사용하지 않습니다.")\n\n\nfill_choice, fill_class = read_fill_choice()\nfilled_lc, egis_fill_stats = apply_egis_fill(egis_major, unknown_mask, fill_choice, fill_class)\nrecord_fill_decision(fill_choice, fill_class, egis_fill_stats)\nprint("EGIS fill stats:", egis_fill_stats)\negis_arr, egis_names = egis_onehot_from_lc(filled_lc)\ncondition = np.concatenate([terrain, albedo, aws, egis_arr], axis=0).astype(np.float32)\ncondition_channels = TERRAIN_BANDS + ALBEDO_BANDS + AWS_BANDS + egis_names\nif condition_channels != CONDITION_CHANNELS:\n    raise RuntimeError("condition channel order mismatch")\nmeta = sample_meta(landsat_info["selected_landsat_date"], row["sun_azimuth"], row["sun_elevation"], ROI_CENTER_LAT, ROI_CENTER_LON)\n\nout = OUT_DIR / f"{RUN_LABEL}_{landsat_info[\'selected_landsat_product_id\']}_condition.npz"\nif out.exists() and not OVERWRITE:\n    print("exists:", out)\nelse:\n    np.savez_compressed(\n        out,\n        condition=condition,\n        condition_channels=np.array(condition_channels),\n        lst_k_raw_direct=lst_k_raw_direct.astype(np.float32),\n        lst_clear_mask_direct=lst_clear_mask_direct.astype(np.uint8),\n        lst_cloud_fill_mask_direct=lst_cloud_fill_mask_direct.astype(np.uint8),\n        meta=meta,\n        meta_channels=np.array(META_CHANNELS),\n        run_label=np.array(RUN_LABEL),\n        roi_label=np.array(ROI_LABEL),\n        data_generation_source=np.array("direct_ee_kma_egis_wms"),\n        requested_date=np.array(landsat_info["requested_date"]),\n        date=np.array(landsat_info["selected_landsat_date"]),\n        selected_landsat_date=np.array(landsat_info["selected_landsat_date"]),\n        landsat_exact_available=np.array(bool(landsat_info["landsat_exact_available"])),\n        landsat_exact_count=np.array(int(landsat_info["landsat_exact_count"])),\n        selected_landsat_product_id=np.array(landsat_info["selected_landsat_product_id"]),\n        selected_landsat_scene_id=np.array(landsat_info["selected_landsat_scene_id"]),\n        selected_landsat_collection=np.array(landsat_info["selected_landsat_collection"]),\n        latitude=np.array(float(ROI_CENTER_LAT), dtype=np.float32),\n        longitude=np.array(float(ROI_CENTER_LON), dtype=np.float32),\n        year=np.array(int(analysis_year)),\n        sun_azimuth=np.array(float(row["sun_azimuth"]), dtype=np.float32),\n        sun_elevation=np.array(float(row["sun_elevation"]), dtype=np.float32),\n        aws_source=np.array("KMA_API_direct"),\n        aws_tm=np.array(str(aws_tm)),\n        aws_station_count=np.array(int(len(aws_station_frame))),\n        terrain_source=np.array("EE_USGS_SRTMGL1_003_direct"),\n        albedo_source=np.array("EE_Landsat_attempt3_clear_mask_focal_mean_local_expanding_mean"),\n        **{k: np.array(v) for k, v in albedo_fill_stats.items()},\n        egis_source=np.array("EGIS_WMS_direct"),\n        egis_policy_year=np.array(int(egis_policy["policy_year"])),\n        egis_source_key=np.array(str(egis_policy["source_key"])),\n        egis_source_level=np.array(str(egis_policy["source_level"])),\n        egis_raw_direct_npz=np.array(str(raw_egis_npz)),\n        egis_unknown_pixels=np.array(int(unknown_mask.sum())),\n        egis_unknown_ratio=np.array(float(unknown_mask.mean()), dtype=np.float32),\n        egis_unknown_fill_choice=np.array("auto_attempt3" if fill_choice == "auto" else str(fill_choice)),\n        egis_unknown_fill_class=np.array(-1),\n        egis_source_nodata_water_fill_mask=np.zeros_like(egis_source_nodata, dtype=np.uint8),\n        egis_unknown_review_png=np.array(str(preview_png)),\n        meta_norm_info=np.array(json.dumps({"source": META_NORM_SOURCE, "lat_mean": LAT_META_MEAN, "lat_std": LAT_META_STD, "lon_mean": LON_META_MEAN, "lon_std": LON_META_STD}, ensure_ascii=False)),\n        **{k: np.array(v) for k, v in egis_unrecognized_fill_stats.items()},\n        **{k: np.array(v) for k, v in egis_fill_stats.items()},\n    )\n    manifest_row = {\n        "output": str(out),\n        "run_label": RUN_LABEL,\n        "roi_label": ROI_LABEL,\n        "data_generation_source": "direct_ee_kma_egis_wms",\n        "requested_date": landsat_info["requested_date"],\n        "selected_landsat_date": landsat_info["selected_landsat_date"],\n        "landsat_exact_available": bool(landsat_info["landsat_exact_available"]),\n        "landsat_exact_count": int(landsat_info["landsat_exact_count"]),\n        "selected_landsat_product_id": landsat_info["selected_landsat_product_id"],\n        "selected_landsat_scene_id": landsat_info["selected_landsat_scene_id"],\n        "lst_clear_pixels": int(lst_clear_mask_direct.sum()),\n        "lst_clear_fraction": float(lst_clear_mask_direct.mean()),\n        "roi_center_lat": float(ROI_CENTER_LAT),\n        "roi_center_lon": float(ROI_CENTER_LON),\n        "aws_source": "KMA_API_direct",\n        "aws_tm": str(aws_tm),\n        "aws_station_count": int(len(aws_station_frame)),\n        "egis_source": "EGIS_WMS_direct",\n        "egis_source_key": egis_policy["source_key"],\n        "egis_unrecognized_pixels": int(egis_unrecognized_mask.sum()),\n        "egis_unknown_pixels": int(unknown_mask.sum()),\n        "egis_unknown_fill_choice": "auto_attempt3" if fill_choice == "auto" else fill_choice,\n        "egis_source_nodata_water_fill_pixels": 0,\n        "unknown_review_png": str(preview_png),\n    }\n    manifest_row.update(albedo_fill_stats)\n    manifest_row.update(egis_unrecognized_fill_stats)\n    manifest_row.update(egis_fill_stats)\n    manifest = pd.DataFrame([manifest_row])\n    manifest.to_csv(OUT_DIR / "condition_manifest.csv", index=False, encoding="utf-8-sig")\n    print("saved condition:", out)\n    print("manifest:", OUT_DIR / "condition_manifest.csv")\nout\n\n\n# Condition 생성 직후 시각화까지 바로 저장/표시한다.\ndef scalar_value(value):\n    arr = np.asarray(value)\n    return arr.item() if arr.shape == () else arr\n\n\ndef colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):\n    import matplotlib.cm as mpl_cm\n    values = np.asarray(arr, dtype=np.float32)\n    finite = np.isfinite(values)\n    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)\n    rgba[..., 3] = 255\n    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    if finite.any():\n        if vmin is None or vmax is None:\n            lo, hi = np.nanpercentile(values[finite], percentiles)\n            if vmin is None:\n                vmin = float(lo)\n            if vmax is None:\n                vmax = float(hi)\n        denom = max(float(vmax - vmin), 1e-6)\n        normed = np.clip((values - float(vmin)) / denom, 0, 1)\n        cmap = mpl_cm.get_cmap(cmap_name)\n        rgba = (cmap(normed) * 255).astype(np.uint8)\n        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    return rgba\n\n\ndef condition_egis_class_map(condition, channels):\n    egis_idx = [i for i, name in enumerate(channels) if str(name).startswith(\'egis_cat_\')]\n    if not egis_idx:\n        return None\n    egis_stack = condition[egis_idx]\n    class_values = []\n    for idx in egis_idx:\n        name = str(channels[idx])\n        class_values.append(int(name.split(\'_\')[2]))\n    winner = np.argmax(egis_stack, axis=0)\n    lc = np.zeros(egis_stack.shape[1:], dtype=np.int16)\n    for k, cls in enumerate(class_values):\n        lc[winner == k] = int(cls)\n    return lc\n\n\ndef final_condition_visualization(npz_path):\n    with np.load(npz_path, allow_pickle=True) as z:\n        cond = z[\'condition\'].astype(np.float32)\n        channels = [str(x) for x in z[\'condition_channels\']]\n        requested = scalar_value(z.get(\'requested_date\', \'\'))\n        selected = scalar_value(z.get(\'selected_landsat_date\', \'\'))\n        roi = scalar_value(z.get(\'roi_label\', ROI_LABEL))\n    tiles = []\n    for name in [\'elevation\', \'slope\', \'hillshade_landsat\', \'albedo\', \'avg_temp\', \'rain_1h\', \'humidity\']:\n        if name in channels:\n            tiles.append((name, colormap_rgba(cond[channels.index(name)], cmap_name=\'turbo\')))\n    lc = condition_egis_class_map(cond, channels)\n    if lc is not None:\n        tiles.append((\'egis filled classes\', hybrid_to_rgba(lc)))\n    if not tiles:\n        raise RuntimeError(\'No visualizable condition channels found\')\n\n    labeled = []\n    for title, rgba in tiles:\n        img = Image.fromarray(rgba).convert(\'RGBA\').resize((256, 256), Image.Resampling.NEAREST)\n        labeled.append(add_label(img, title))\n    cols = 4\n    rows = int(math.ceil(len(labeled) / cols))\n    title_h = 30\n    sheet = Image.new(\'RGBA\', (cols * 256, rows * 280 + title_h), (255, 255, 255, 255))\n    draw = ImageDraw.Draw(sheet)\n    draw.text((6, 8), f\'ROI={roi} | requested={requested} | selected_landsat={selected}\', fill=(0, 0, 0, 255))\n    for i, img in enumerate(labeled):\n        sheet.paste(img, ((i % cols) * 256, title_h + (i // cols) * 280))\n    quicklook = VIS_DIR / f\'{RUN_LABEL}_condition_quicklook.png\'\n    sheet.save(quicklook)\n    return quicklook\n\ncondition_quicklook_png = final_condition_visualization(out)\nprint(\'condition quicklook:\', condition_quicklook_png)\ntry:\n    from IPython.display import Image as DisplayImage, display\n    display(DisplayImage(filename=str(condition_quicklook_png)))\nexcept Exception:\n    pass\n'

_INTERNAL_RECONSTRUCTION_GENERATION_CODE = 'from pathlib import Path\nimport json, math\nfrom datetime import datetime\nfrom zoneinfo import ZoneInfo\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\n\n\n# 01에서 만든 같은 RUN_ROOT_DIR/condition 결과를 그대로 이어서 복원한다.\nCKPT_NAME = "best_balanced.pt"\nUSE_PRIOR_MEAN = True\nN_RANDOM_SAMPLES = 0        # 0이면 prior mean만 저장. 불확실성 샘플이 필요하면 8 등으로 설정\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\n\nCONDITION_DIR = RUN_ROOT_DIR / "outputs" / "condition"\nSTRUCTURED_DIR = CONDITION_DIR\nRECONSTRUCTION_DIR = RUN_ROOT_DIR / "outputs" / "reconstruction"\nOUT_DIR = RECONSTRUCTION_DIR\nVIS_DIR = RUN_ROOT_DIR / "outputs" / "visualization"\nCKPT_PATH = RECONST_DIR / "savedpt" / CKPT_NAME\nfor p in [OUT_DIR, VIS_DIR]:\n    p.mkdir(parents=True, exist_ok=True)\nif not CKPT_PATH.exists():\n    raise FileNotFoundError(f"checkpoint missing: {CKPT_PATH}")\nprint("run_label:", RUN_LABEL)\nprint("run root:", RUN_ROOT_DIR)\nprint("device:", DEVICE)\nprint("structured:", STRUCTURED_DIR)\nprint("reconstruction output:", OUT_DIR)\nprint("visualization output:", VIS_DIR)\nprint("checkpoint:", CKPT_PATH)\n\n\ndef scalar(value):\n    arr = np.asarray(value)\n    return arr.item() if arr.shape == () else arr\n\ndef scalar_str(value):\n    return str(scalar(value))\n\ndef minmax(values, channel_stats, names):\n    out = values.astype(np.float32, copy=True)\n    for i, name in enumerate(names):\n        lo = channel_stats[name]["min"]\n        hi = channel_stats[name]["max"]\n        denom = hi - lo\n        out[i] = 0.0 if abs(denom) <= 1e-12 else (out[i] - lo) / denom\n    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)\n\ndef denormalize(values, channel_stats, names):\n    out = values.astype(np.float32, copy=True)\n    for i, name in enumerate(names):\n        lo = channel_stats[name]["min"]\n        hi = channel_stats[name]["max"]\n        out[i] = out[i] * (hi - lo) + lo\n    return out.astype(np.float32)\n\n\ndef channel_indices(names, predicates):\n    out = []\n    for i, name in enumerate(names):\n        low = name.lower()\n        if any(pred(low) for pred in predicates):\n            out.append(i)\n    return out\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, padding=1),\n            nn.GroupNorm(min(8, out_ch), out_ch),\n            nn.SiLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, padding=1),\n            nn.GroupNorm(min(8, out_ch), out_ch),\n            nn.SiLU(inplace=True),\n        )\n    def forward(self, x): return self.net(x)\n\nclass Down(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.net = nn.Sequential(nn.Conv2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))\n    def forward(self, x): return self.net(x)\n\nclass Up(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.net = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))\n    def forward(self, x): return self.net(x)\n\nclass FiLM(nn.Module):\n    def __init__(self, context_dim, ch):\n        super().__init__()\n        self.to_gamma_beta = nn.Linear(context_dim, ch * 2)\n    def forward(self, x, context):\n        gb = self.to_gamma_beta(context)\n        gamma, beta = gb.chunk(2, dim=1)\n        return x * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]\n\nclass ConditionEncoder(nn.Module):\n    def __init__(self, in_ch, width):\n        super().__init__()\n        self.s0 = ConvBlock(in_ch, width)\n        self.d1 = Down(width, width)\n        self.d2 = Down(width, width * 2)\n        self.d3 = Down(width * 2, width * 4)\n        self.d4 = Down(width * 4, width * 8)\n        self.d5 = Down(width * 8, width * 8)\n    def forward(self, x):\n        s0 = self.s0(x); s1 = self.d1(s0); s2 = self.d2(s1); s3 = self.d3(s2); s4 = self.d4(s3); b = self.d5(s4)\n        return b, [s0, s1, s2, s3, s4]\n\nclass Decoder(nn.Module):\n    def __init__(self, bottleneck_ch, out_ch, width, context_dim, latent_ch=0, gated_skips=True):\n        super().__init__()\n        self.gated_skips = gated_skips\n        in_ch = bottleneck_ch + latent_ch\n        self.u4 = Up(in_ch, width * 8); self.f4 = ConvBlock(width * 16, width * 8); self.film4 = FiLM(context_dim, width * 8)\n        self.u3 = Up(width * 8, width * 4); self.f3 = ConvBlock(width * 8, width * 4); self.film3 = FiLM(context_dim, width * 4)\n        self.u2 = Up(width * 4, width * 2); self.f2 = ConvBlock(width * 4, width * 2); self.film2 = FiLM(context_dim, width * 2)\n        self.u1 = Up(width * 2, width); self.f1 = ConvBlock(width * 2, width); self.film1 = FiLM(context_dim, width)\n        self.u0 = Up(width, width); self.f0 = ConvBlock(width * 2, width); self.film0 = FiLM(context_dim, width)\n        self.out = nn.Conv2d(width, out_ch, 3, padding=1)\n        if gated_skips:\n            self.skip_gates = nn.Parameter(torch.full((5,), -0.5))\n    def gate(self, skip, idx):\n        if not self.gated_skips:\n            return skip\n        return skip * torch.sigmoid(self.skip_gates[idx])\n    def forward(self, bottleneck, skips, context, latent_map=None):\n        x = torch.cat([bottleneck, latent_map], dim=1) if latent_map is not None else bottleneck\n        s0, s1, s2, s3, s4 = skips\n        x = self.u4(x); x = self.f4(torch.cat([x, self.gate(s4, 4)], dim=1)); x = self.film4(x, context)\n        x = self.u3(x); x = self.f3(torch.cat([x, self.gate(s3, 3)], dim=1)); x = self.film3(x, context)\n        x = self.u2(x); x = self.f2(torch.cat([x, self.gate(s2, 2)], dim=1)); x = self.film2(x, context)\n        x = self.u1(x); x = self.f1(torch.cat([x, self.gate(s1, 1)], dim=1)); x = self.film1(x, context)\n        x = self.u0(x); x = self.f0(torch.cat([x, self.gate(s0, 0)], dim=1)); x = self.film0(x, context)\n        return self.out(x)\n\nclass PosteriorEncoder(nn.Module):\n    def __init__(self, in_ch, width):\n        super().__init__()\n        self.net = nn.Sequential(ConvBlock(in_ch, width), Down(width, width), Down(width, width * 2), Down(width * 2, width * 4), Down(width * 4, width * 8), Down(width * 8, width * 8))\n    def forward(self, condition, target):\n        return self.net(torch.cat([condition, target], dim=1))\n\nclass GaussianHead(nn.Module):\n    def __init__(self, in_dim, hidden_dim, latent_dim):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.SiLU(inplace=True), nn.Linear(hidden_dim, hidden_dim), nn.SiLU(inplace=True))\n        self.mu = nn.Linear(hidden_dim, latent_dim)\n        self.logvar = nn.Linear(hidden_dim, latent_dim)\n    def forward(self, x):\n        h = self.net(x)\n        return self.mu(h), self.logvar(h).clamp(-12, 8)\n\nclass MultiLatentConditionalPriorResidualCVAE(nn.Module):\n    def __init__(self, condition_ch, target_ch, meta_dim, latent_blocks, width=32, grid_n=256, condition_channels=None, meta_channels=None):\n        super().__init__()\n        condition_channels = condition_channels or []\n        meta_channels = meta_channels or []\n        self.weather_idx = channel_indices(condition_channels, [lambda s: s in {\'avg_temp\', \'humidity\', \'wind_u\', \'wind_v\', \'rain_1h\'}])\n        self.land_idx = channel_indices(condition_channels, [lambda s: s == \'albedo\', lambda s: s.startswith(\'egis_cat_\')])\n        self.static_idx = channel_indices(condition_channels, [lambda s: s in {\'elevation\', \'slope\', \'aspect_sin\', \'aspect_cos\', \'hillshade_landsat\'}])\n        meta_index = {name: i for i, name in enumerate(meta_channels)}\n        self.season_meta_idx = [meta_index[x] for x in [\'day_sin\', \'day_cos\', \'sun_azimuth_norm\', \'sun_elevation_norm\'] if x in meta_index]\n        self.global_meta_idx = [meta_index[x] for x in [\'lat_norm\', \'lon_norm\'] if x in meta_index]\n        reduced = grid_n // 32\n        flat = width * 8 * reduced * reduced\n        self.latent_blocks = dict(latent_blocks)\n        self.latent_dim = sum(self.latent_blocks.values())\n        self.width = width\n        self.reduced = reduced\n        self.condition_encoder = ConditionEncoder(condition_ch, width)\n        self.posterior_encoder = PosteriorEncoder(condition_ch + target_ch, width)\n        self.context_head = nn.Sequential(nn.Linear(width * 8 + meta_dim, width * 8), nn.SiLU(inplace=True), nn.Linear(width * 8, width * 8), nn.SiLU(inplace=True))\n        pooled_dim = width * 8\n        self.block_input_dims = {\n            \'global\': pooled_dim + len(self.static_idx) + len(self.global_meta_idx),\n            \'season\': pooled_dim + len(self.season_meta_idx),\n            \'weather\': pooled_dim + len(self.weather_idx) + len(self.season_meta_idx),\n            \'land\': pooled_dim + len(self.land_idx) + len(self.global_meta_idx),\n            \'residual\': pooled_dim + meta_dim,\n        }\n        post_shared_dim = width * 16\n        self.post_shared = nn.Sequential(nn.Linear(flat + meta_dim, post_shared_dim), nn.SiLU(inplace=True), nn.Linear(post_shared_dim, post_shared_dim), nn.SiLU(inplace=True))\n        self.prior_heads = nn.ModuleDict({name: GaussianHead(self.block_input_dims[name], width * 8, dim) for name, dim in self.latent_blocks.items()})\n        self.posterior_heads = nn.ModuleDict({name: GaussianHead(post_shared_dim + self.block_input_dims[name], width * 8, dim) for name, dim in self.latent_blocks.items()})\n        self.z_to_map = nn.Sequential(nn.Linear(self.latent_dim + width * 8, flat), nn.SiLU(inplace=True))\n        self.base_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=0, gated_skips=True)\n        self.residual_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=width * 8, gated_skips=True)\n    def channel_summary(self, condition, idxs):\n        if not idxs:\n            return condition.new_zeros((condition.shape[0], 0))\n        return condition[:, idxs].mean(dim=(2, 3))\n    def meta_select(self, meta, idxs):\n        if not idxs:\n            return meta.new_zeros((meta.shape[0], 0))\n        return meta[:, idxs]\n    def block_inputs(self, condition, bottleneck, meta):\n        pooled = bottleneck.mean(dim=(2, 3))\n        return {\n            \'global\': torch.cat([pooled, self.channel_summary(condition, self.static_idx), self.meta_select(meta, self.global_meta_idx)], dim=1),\n            \'season\': torch.cat([pooled, self.meta_select(meta, self.season_meta_idx)], dim=1),\n            \'weather\': torch.cat([pooled, self.channel_summary(condition, self.weather_idx), self.meta_select(meta, self.season_meta_idx)], dim=1),\n            \'land\': torch.cat([pooled, self.channel_summary(condition, self.land_idx), self.meta_select(meta, self.global_meta_idx)], dim=1),\n            \'residual\': torch.cat([pooled, meta], dim=1),\n        }\n    def context(self, bottleneck, meta):\n        pooled = bottleneck.mean(dim=(2, 3))\n        return self.context_head(torch.cat([pooled, meta], dim=1))\n    def prior(self, condition, bottleneck, meta):\n        inputs = self.block_inputs(condition, bottleneck, meta)\n        return {name: self.prior_heads[name](inputs[name]) for name in self.latent_blocks}\n    def posterior(self, condition, target, bottleneck, meta):\n        post_feat = self.posterior_encoder(condition, target).flatten(1)\n        shared = self.post_shared(torch.cat([post_feat, meta], dim=1))\n        inputs = self.block_inputs(condition, bottleneck, meta)\n        return {name: self.posterior_heads[name](torch.cat([shared, inputs[name]], dim=1)) for name in self.latent_blocks}\n    def reparameterize(self, mu, logvar):\n        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)\n    def sample_z(self, stats, use_mean=False):\n        parts = []\n        for name in self.latent_blocks:\n            mu, logvar = stats[name]\n            parts.append(mu if use_mean else self.reparameterize(mu, logvar))\n        return torch.cat(parts, dim=1)\n    def decode(self, condition, meta, z=None, use_prior_mean=False):\n        b, skips = self.condition_encoder(condition)\n        ctx = self.context(b, meta)\n        prior_stats = self.prior(condition, b, meta)\n        if z is None:\n            z = self.sample_z(prior_stats, use_mean=use_prior_mean)\n        zmap = self.z_to_map(torch.cat([z, ctx], dim=1)).view(condition.shape[0], self.width * 8, self.reduced, self.reduced)\n        base = self.base_decoder(b, skips, ctx)\n        residual = self.residual_decoder(b, skips, ctx, zmap)\n        return base + residual, base, residual, prior_stats\n\n\ndef load_model_from_ckpt(ckpt_path):\n    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)\n    condition_channels = [str(x) for x in ckpt["condition_channels"]]\n    target_channels = [str(x) for x in ckpt["target_channels"]]\n    meta_channels = [str(x) for x in ckpt["meta_channels"]]\n    latent_blocks = ckpt.get("latent_blocks", {"global": 24, "season": 16, "weather": 16, "land": 24, "residual": 16})\n    width = int(ckpt.get("width", 32))\n    grid_n = int(ckpt.get("grid_n", 256))\n    model = MultiLatentConditionalPriorResidualCVAE(\n        len(condition_channels), len(target_channels), len(meta_channels),\n        latent_blocks, width=width, grid_n=grid_n,\n        condition_channels=condition_channels, meta_channels=meta_channels,\n    ).to(DEVICE)\n    model.load_state_dict(ckpt["model"])\n    model.eval()\n    print("loaded:", ckpt_path.name, "epoch=", ckpt.get("epoch"), "metrics=", ckpt.get("metrics", {}))\n    return ckpt, model\n\nckpt, model = load_model_from_ckpt(CKPT_PATH)\n\n\ndef read_condition_for_ckpt(path, ckpt):\n    stats = ckpt["normalization_stats"]\n    condition_channels = [str(x) for x in ckpt["condition_channels"]]\n    meta_channels = [str(x) for x in ckpt["meta_channels"]]\n    with np.load(path, allow_pickle=True) as z:\n        z_condition_channels = [str(x) for x in z["condition_channels"]]\n        z_meta_channels = [str(x) for x in z["meta_channels"]]\n        if z_condition_channels != condition_channels:\n            raise RuntimeError(f"condition channel mismatch: {path.name}")\n        if z_meta_channels != meta_channels:\n            raise RuntimeError(f"meta channel mismatch: {path.name}")\n        condition = minmax(z["condition"].astype(np.float32), stats["condition"], condition_channels)\n        meta = z["meta"].astype(np.float32)\n        attrs = {k: z[k] for k in z.files if k not in {"condition", "condition_channels", "meta", "meta_channels"}}\n    return torch.from_numpy(condition[None]).float(), torch.from_numpy(meta[None]).float(), attrs\n\ndef colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):\n    import matplotlib.cm as mpl_cm\n    values = np.asarray(arr, dtype=np.float32)\n    finite = np.isfinite(values)\n    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)\n    rgba[..., 3] = 255\n    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    if finite.any():\n        if vmin is None or vmax is None:\n            lo, hi = np.nanpercentile(values[finite], percentiles)\n            if vmin is None:\n                vmin = float(lo)\n            if vmax is None:\n                vmax = float(hi)\n        denom = max(float(vmax - vmin), 1e-6)\n        normed = np.clip((values - float(vmin)) / denom, 0, 1)\n        cmap = mpl_cm.get_cmap(cmap_name)\n        rgba = (cmap(normed) * 255).astype(np.uint8)\n        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    return rgba\n\n\ndef heatmap_rgba(arr):\n    return colormap_rgba(arr, cmap_name="turbo")\n\ndef add_label(img, label, height=24):\n    from PIL import Image, ImageDraw\n    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))\n    base.paste(img, (0, height))\n    draw = ImageDraw.Draw(base)\n    draw.text((4, 5), str(label)[:80], fill=(0, 0, 0, 255))\n    return base\n\ndef save_prediction_png(pred, target_channels, out_png, title):\n    from PIL import Image, ImageDraw\n    tiles = []\n    for name, arr in zip(target_channels, pred):\n        img = Image.fromarray(heatmap_rgba(arr)).resize((256, 256), Image.Resampling.NEAREST)\n        tiles.append(add_label(img, name))\n    title_h = 28\n    sheet = Image.new("RGBA", (256 * len(tiles), 256 + 24 + title_h), (255, 255, 255, 255))\n    draw = ImageDraw.Draw(sheet)\n    draw.text((4, 6), str(title)[:140], fill=(0, 0, 0, 255))\n    for i, tile in enumerate(tiles):\n        sheet.paste(tile, (i * 256, title_h))\n    sheet.save(out_png)\n\ndef clear_preserved_outputs(pred, attrs, min_clear_pixels=512):\n    raw = np.asarray(attrs.get("lst_k_raw_direct"), dtype=np.float32) if "lst_k_raw_direct" in attrs else None\n    clear = np.asarray(attrs.get("lst_clear_mask_direct"), dtype=bool) if "lst_clear_mask_direct" in attrs else None\n    if raw is None or clear is None:\n        return pred.astype(np.float32), pred.astype(np.float32), pred.astype(np.float32), np.float32(0.0), np.array(False)\n    valid_clear = clear & np.isfinite(raw)\n    bias = np.float32(0.0)\n    used_bias = np.array(False)\n    if pred.shape[0] > 0 and int(valid_clear.sum()) >= int(min_clear_pixels):\n        diff = pred[0][valid_clear] - raw[valid_clear]\n        diff = diff[np.isfinite(diff)]\n        if diff.size >= int(min_clear_pixels):\n            bias = np.float32(np.nanmedian(diff))\n            used_bias = np.array(True)\n    corrected = (pred - bias).astype(np.float32)\n    clear_preserved = pred.astype(np.float32).copy()\n    corrected_clear_preserved = corrected.copy()\n    for i in range(pred.shape[0]):\n        clear_preserved[i][valid_clear] = raw[valid_clear]\n        corrected_clear_preserved[i][valid_clear] = raw[valid_clear]\n    return clear_preserved, corrected, corrected_clear_preserved, bias, used_bias\n\ndef run_one(path):\n    out_npz = OUT_DIR / f"{path.stem}_lst_reconst.npz"\n    out_png = OUT_DIR / f"{path.stem}_lst_reconst.png"\n    if out_npz.exists() and not OVERWRITE:\n        return {"condition": str(path), "output": str(out_npz), "png": str(out_png), "status": "SKIP_EXISTING"}\n    cond, meta, attrs = read_condition_for_ckpt(path, ckpt)\n    cond = cond.to(DEVICE)\n    meta = meta.to(DEVICE)\n    with torch.no_grad():\n        pred_n, base_n, residual_n, prior_stats = model.decode(cond, meta, use_prior_mean=USE_PRIOR_MEAN)\n        pred = denormalize(pred_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]])\n        base = denormalize(base_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]])\n        residual = residual_n.cpu().numpy()[0].astype(np.float32)\n        random_preds = []\n        for _ in range(int(N_RANDOM_SAMPLES)):\n            sample_n, _, _, _ = model.decode(cond, meta, use_prior_mean=False)\n            random_preds.append(denormalize(sample_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]]))\n    target_channels = np.array([str(x) for x in ckpt["target_channels"]])\n    payload = dict(attrs)\n    clear_preserved, bias_corrected, bias_corrected_clear_preserved, clear_bias_k, clear_bias_used = clear_preserved_outputs(pred, payload)\n    save_prediction_png(bias_corrected_clear_preserved, target_channels, out_png, f"final clear-preserved | roi={scalar_str(attrs.get(\'roi_label\', \'\'))} | requested={scalar_str(attrs.get(\'requested_date\', \'\'))} | selected_landsat={scalar_str(attrs.get(\'selected_landsat_date\', \'\'))}")\n    payload.update({\n        "prediction": pred.astype(np.float32),\n        "prediction_clear_preserved": clear_preserved.astype(np.float32),\n        "prediction_bias_corrected": bias_corrected.astype(np.float32),\n        "prediction_bias_corrected_clear_preserved": bias_corrected_clear_preserved.astype(np.float32),\n        "clear_pixel_model_bias_k": np.array(float(clear_bias_k), dtype=np.float32),\n        "clear_pixel_bias_correction_used": np.array(bool(clear_bias_used)),\n        "base": base.astype(np.float32),\n        "residual_normalized": residual.astype(np.float32),\n        "target_channels": target_channels,\n        "checkpoint": np.array(str(CKPT_PATH)),\n        "use_prior_mean": np.array(bool(USE_PRIOR_MEAN)),\n    })\n    if random_preds:\n        payload["random_predictions"] = np.stack(random_preds).astype(np.float32)\n    np.savez_compressed(out_npz, **payload)\n    return {\n        "condition": str(path),\n        "output": str(out_npz),\n        "png": str(out_png),\n        "status": "OK",\n        "run_label": scalar_str(attrs.get("run_label", RUN_LABEL)),\n        "data_generation_source": scalar_str(attrs.get("data_generation_source", "")),\n        "roi_label": scalar_str(attrs.get("roi_label", "")),\n        "requested_date": scalar_str(attrs.get("requested_date", "")),\n        "landsat_exact_available": scalar_str(attrs.get("landsat_exact_available", "")),\n        "landsat_exact_count": scalar_str(attrs.get("landsat_exact_count", "")),\n        "selected_landsat_date": scalar_str(attrs.get("selected_landsat_date", "")),\n        "template_date": scalar_str(attrs.get("template_date", "")),\n        "template_date_delta_days": scalar_str(attrs.get("template_date_delta_days", "")),\n        "clear_pixel_model_bias_k": float(clear_bias_k),\n        "clear_pixel_bias_correction_used": bool(clear_bias_used),\n    }\n\ncondition_files = sorted(STRUCTURED_DIR.glob("*_condition.npz"))\nif not condition_files:\n    raise FileNotFoundError(f"condition npz missing: {STRUCTURED_DIR}. Run 01 first.")\nrows = [run_one(path) for path in condition_files]\nmanifest = pd.DataFrame(rows)\nmanifest.to_csv(OUT_DIR / "reconstruction_manifest.csv", index=False, encoding="utf-8-sig")\nprint(manifest.to_string(index=False))\nprint("manifest:", OUT_DIR / "reconstruction_manifest.csv")\n\n\n# Reconstruction 생성 직후 quicklook까지 바로 저장한다.\ndef scalar_value(value):\n    arr = np.asarray(value)\n    return arr.item() if arr.shape == () else arr\n\n\ndef npz_scalar(z, key, default=""):\n    return scalar_value(z[key]) if key in z.files else default\n\n\ndef colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):\n    import matplotlib.cm as mpl_cm\n    values = np.asarray(arr, dtype=np.float32)\n    finite = np.isfinite(values)\n    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)\n    rgba[..., 3] = 255\n    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    if finite.any():\n        if vmin is None or vmax is None:\n            lo, hi = np.nanpercentile(values[finite], percentiles)\n            if vmin is None:\n                vmin = float(lo)\n            if vmax is None:\n                vmax = float(hi)\n        denom = max(float(vmax - vmin), 1e-6)\n        normed = np.clip((values - float(vmin)) / denom, 0, 1)\n        cmap = mpl_cm.get_cmap(cmap_name)\n        rgba = (cmap(normed) * 255).astype(np.uint8)\n        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)\n    return rgba\n\n\ndef lst_rgba(arr, vmin=None, vmax=None):\n    return colormap_rgba(arr, cmap_name="turbo", vmin=vmin, vmax=vmax)\n\n\ndef diff_rgba(arr, limit=6.0):\n    return colormap_rgba(arr, cmap_name="coolwarm", vmin=-float(limit), vmax=float(limit))\n\n\ndef shared_limits(arrays, percentiles=(2, 98), min_span=2.0):\n    vals = []\n    for arr in arrays:\n        values = np.asarray(arr, dtype=np.float32)\n        finite = np.isfinite(values)\n        if bool(finite.any()):\n            vals.append(values[finite].reshape(-1))\n    if not vals:\n        return None, None\n    vals = np.concatenate(vals)\n    lo, hi = np.nanpercentile(vals, percentiles)\n    if float(hi - lo) < float(min_span):\n        mid = 0.5 * float(lo + hi)\n        lo = mid - float(min_span) / 2\n        hi = mid + float(min_span) / 2\n    return float(lo), float(hi)\n\n\ndef diff_limit(arr, default=6.0, min_limit=3.0, max_limit=15.0):\n    values = np.asarray(arr, dtype=np.float32)\n    finite = np.isfinite(values)\n    if not bool(finite.any()):\n        return float(default)\n    lim = float(np.nanpercentile(np.abs(values[finite]), 98))\n    return float(np.clip(lim, min_limit, max_limit))\n\n\ndef error_label(name, diff):\n    finite = np.isfinite(diff)\n    if not bool(finite.any()):\n        return f"{name} | no common clear pixels"\n    vals = diff[finite]\n    rmse = float(np.sqrt(np.nanmean(vals ** 2)))\n    mae = float(np.nanmean(np.abs(vals)))\n    bias = float(np.nanmean(vals))\n    return f"{name} | RMSE={rmse:.2f}K MAE={mae:.2f}K bias={bias:.2f}K n={int(finite.sum())}"\n\n\ndef add_label_local(img, label, height=24):\n    from PIL import Image, ImageDraw\n    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))\n    base.paste(img, (0, height))\n    draw = ImageDraw.Draw(base)\n    draw.text((4, 5), str(label)[:90], fill=(0, 0, 0, 255))\n    return base\n\n\ndef lst_label(name, arr):\n    finite = np.isfinite(arr)\n    if finite.any():\n        return f"{name} | min={np.nanmin(arr):.2f}K max={np.nanmax(arr):.2f}K valid={int(finite.sum())}"\n    return f"{name} | no valid pixels"\n\n\ndef build_reconstruction_quicklook(npz_path):\n    from PIL import Image, ImageDraw\n    npz_path = Path(npz_path)\n    with np.load(npz_path, allow_pickle=True) as z:\n        pred = z["prediction"].astype(np.float32)\n        final_pred = z["prediction_bias_corrected_clear_preserved"].astype(np.float32) if "prediction_bias_corrected_clear_preserved" in z.files else z["prediction_clear_preserved"].astype(np.float32) if "prediction_clear_preserved" in z.files else pred\n        target_channels = [str(x) for x in z["target_channels"]]\n        requested = npz_scalar(z, "requested_date", "")\n        selected = npz_scalar(z, "selected_landsat_date", "")\n        roi = npz_scalar(z, "roi_label", ROI_LABEL)\n        product = npz_scalar(z, "selected_landsat_product_id", "")\n        raw_lst = z["lst_k_raw_direct"].astype(np.float32) if "lst_k_raw_direct" in z.files else None\n        raw_clear = z["lst_clear_mask_direct"].astype(bool) if "lst_clear_mask_direct" in z.files else None\n        bias = float(npz_scalar(z, "clear_pixel_model_bias_k", 0.0))\n        bias_used = bool(npz_scalar(z, "clear_pixel_bias_correction_used", False))\n\n    raw_display = None\n    if raw_lst is not None:\n        raw_display = raw_lst.copy()\n        if raw_clear is not None:\n            raw_display[~raw_clear] = np.nan\n    vmin, vmax = shared_limits(([raw_display] if raw_display is not None else []) + [pred[0], final_pred[0]])\n\n    tiles = []\n    for i, name in enumerate(target_channels):\n        img = Image.fromarray(lst_rgba(final_pred[i], vmin=vmin, vmax=vmax)).resize((256, 256), Image.Resampling.NEAREST)\n        tiles.append(add_label_local(img, lst_label(f"final clear-preserved {name}", final_pred[i])))\n        img = Image.fromarray(lst_rgba(pred[i], vmin=vmin, vmax=vmax)).resize((256, 256), Image.Resampling.NEAREST)\n        tiles.append(add_label_local(img, lst_label(f"model prior {name}", pred[i])))\n        if raw_display is not None and i == 0:\n            diff = np.where(np.isfinite(raw_display), pred[i] - raw_display, np.nan).astype(np.float32)\n            lim = diff_limit(diff)\n            img = Image.fromarray(diff_rgba(diff, limit=lim)).resize((256, 256), Image.Resampling.NEAREST)\n            tiles.append(add_label_local(img, error_label(f"model prior - observed clear", diff)))\n    cols = min(3, max(1, len(tiles)))\n    rows = int(np.ceil(len(tiles) / cols))\n    title_h = 42\n    sheet = Image.new("RGBA", (cols * 256, rows * 280 + title_h), (255, 255, 255, 255))\n    draw = ImageDraw.Draw(sheet)\n    scale_text = "shared turbo scale" if vmin is None else f"shared turbo scale {vmin:.1f}-{vmax:.1f}K"\n    draw.text((6, 8), f"ROI={roi} | requested={requested} | selected_landsat={selected} | {product}", fill=(0, 0, 0, 255))\n    draw.text((6, 24), scale_text + f" | final keeps observed clear pixels | prior bias={bias:.2f}K used={bias_used}", fill=(0, 0, 0, 255))\n    for i, img in enumerate(tiles):\n        sheet.paste(img, ((i % cols) * 256, title_h + (i // cols) * 280))\n    quicklook = VIS_DIR / f"{npz_path.stem}_quicklook.png"\n    sheet.save(quicklook)\n    return quicklook\n\n\ndef final_reconstruction_visualization(manifest_path):\n    df = pd.read_csv(manifest_path)\n    ok = df[df["status"].astype(str).isin(["OK", "SKIP_EXISTING"])].copy()\n    if ok.empty:\n        raise RuntimeError(f"No reconstruction outputs in {manifest_path}")\n    quicklooks = []\n    for _, item in ok.iterrows():\n        quicklooks.append(build_reconstruction_quicklook(Path(item["output"])))\n    return quicklooks\n\nreconstruction_manifest_path = OUT_DIR / "reconstruction_manifest.csv"\nreconstruction_quicklook_pngs = final_reconstruction_visualization(reconstruction_manifest_path)\ntry:\n    from IPython.display import Image as DisplayImage, display\nexcept Exception:\n    DisplayImage = None\n    display = None\nfor quicklook in reconstruction_quicklook_pngs:\n    print("reconstruction quicklook:", quicklook)\n    if display is not None:\n        display(DisplayImage(filename=str(quicklook)))\n'


In [ ]:
def scene_payload(row):
    return {
        "date": str(row["date"]),
        "timestamp_kst": str(row["timestamp_kst"]),
        "scene_id": str(row["scene_id"]),
        "collection": str(row["collection"]),
        "landsat_product_id": str(row["landsat_product_id"]),
        "cloud_cover_scene": None if pd.isna(row["cloud_cover_scene"]) else float(row["cloud_cover_scene"]),
        "sun_azimuth": None if pd.isna(row["sun_azimuth"]) else float(row["sun_azimuth"]),
        "sun_elevation": None if pd.isna(row["sun_elevation"]) else float(row["sun_elevation"]),
    }


EXECUTION_COLUMNS = [
    "batch_index", "period_group", "season_group", "season_name_ko", "date",
    "landsat_product_id", "scene_id", "status", "error", "run_root",
    "clear_pixel_count", "mae_k", "runner_id", "started_kst", "finished_kst",
]
COMBINED_MANIFEST_STATUSES = {"OK", "ERROR"}


def safe_task_token(value):
    token = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")
    return token or "none"


def task_key(row):
    return "__".join([
        safe_task_token(row["period_group"]),
        safe_task_token(row["season_group"]),
        safe_task_token(row["date"]),
        safe_task_token(row["landsat_product_id"]),
        safe_task_token(row["scene_id"]),
    ])


def task_status_path(row):
    return TASK_STATUS_DIR / f"{task_key(row)}.json"



def reconstruction_npz_paths_for_run(run_root):
    run_root = Path(run_root)
    return sorted((run_root / "outputs" / "reconstruction").glob("*_lst_reconst.npz"))


def mae_for_reconstruction_npz(npz_path):
    with np.load(npz_path, allow_pickle=True) as z:
        if "prediction" not in z.files or "lst_k_raw_direct" not in z.files or "lst_clear_mask_direct" not in z.files:
            return None
        pred = z["prediction"].astype(np.float32)
        raw = z["lst_k_raw_direct"].astype(np.float32)
        clear = z["lst_clear_mask_direct"].astype(bool)
        pred_lst = pred[0] if pred.ndim == 3 else pred
        mask = clear & np.isfinite(raw) & np.isfinite(pred_lst)
        n = int(mask.sum())
        if n == 0:
            return None
        diff = pred_lst[mask] - raw[mask]
        return float(np.mean(np.abs(diff))), n


def mae_for_run(run_root):
    abs_error_sum = 0.0
    clear_pixel_count = 0
    for npz_path in reconstruction_npz_paths_for_run(run_root):
        with np.load(npz_path, allow_pickle=True) as z:
            if "prediction" not in z.files or "lst_k_raw_direct" not in z.files or "lst_clear_mask_direct" not in z.files:
                continue
            pred = z["prediction"].astype(np.float32)
            raw = z["lst_k_raw_direct"].astype(np.float32)
            clear = z["lst_clear_mask_direct"].astype(bool)
            pred_lst = pred[0] if pred.ndim == 3 else pred
            mask = clear & np.isfinite(raw) & np.isfinite(pred_lst)
            n = int(mask.sum())
            if n == 0:
                continue
            diff = pred_lst[mask] - raw[mask]
            abs_error_sum += float(np.sum(np.abs(diff)))
            clear_pixel_count += n
    if clear_pixel_count == 0:
        return np.nan, 0
    return abs_error_sum / clear_pixel_count, clear_pixel_count



def write_csv_atomic(frame, csv_path, required=False):
    csv_path = Path(csv_path)
    last_exc = None
    for attempt in range(8):
        tmp_path = csv_path.with_name(f".{csv_path.name}.{RUNNER_ID_SAFE}.{os.getpid()}.{time.time_ns()}.tmp")
        try:
            frame.to_csv(tmp_path, index=False, encoding="utf-8-sig")
            os.replace(tmp_path, csv_path)
            return True
        except PermissionError as exc:
            last_exc = exc
            try:
                if tmp_path.exists():
                    tmp_path.unlink()
            except Exception:
                pass
            time.sleep(0.25 * (attempt + 1))
    if required and last_exc is not None:
        raise last_exc
    print(f"  warning: skipped locked CSV update: {csv_path}")
    return False


def write_runner_execution_manifest(rows):
    write_csv_atomic(pd.DataFrame(rows, columns=EXECUTION_COLUMNS), RUNNER_EXECUTION_MANIFEST_CSV)



def write_task_status(row_dict):
    status_path = TASK_STATUS_DIR / f"{row_dict['task_key']}.json"
    tmp_path = TASK_STATUS_DIR / f".{row_dict['task_key']}.{RUNNER_ID_SAFE}.{os.getpid()}.{time.time_ns()}.tmp"
    tmp_path.write_text(json.dumps(row_dict, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp_path, status_path)
    # Parallel runners do not update shared CSVs. Status JSON is the
    # authoritative per-scene record; merged CSVs are rebuilt after runners finish.


def run_scene_in_current_notebook(row):
    payload = scene_payload(row)
    group = str(row["period_group"])
    season = str(row["season_group"])
    run_ns = dict(globals())
    run_ns.update({
        "TARGET_DATE": payload["date"],
        "ROI_LABEL": ROI_LABEL,
        "ROI_CENTER_LAT": float(ROI_CENTER_LAT),
        "ROI_CENTER_LON": float(ROI_CENTER_LON),
        "RUNS_ROOT_DIR": str(SEASON_RUNS_DIRS[group][season]),
        "PRESELECTED_LANDSAT_SCENE": payload,
        "OVERWRITE": bool(OVERWRITE),
    })

    exec(_INTERNAL_CONDITION_GENERATION_CODE, run_ns)
    exec(_INTERNAL_RECONSTRUCTION_GENERATION_CODE, run_ns)
    return {
        "run_root": str(run_ns.get("RUN_ROOT_DIR", "")),
        "condition_manifest": str(run_ns.get("CONDITION_DIR", "")),
        "reconstruction_manifest": str(run_ns.get("RECONSTRUCTION_DIR", "")),
    }


if not EXECUTE_BATCH:
    print("EXECUTE_BATCH=False. Scene manifest만 생성하고 멈춥니다.")
else:
    exec_rows = []
    write_runner_execution_manifest(exec_rows)
    runner_scene_manifest = scene_manifest.reset_index(drop=True)
    print(f"runner full list: {len(runner_scene_manifest)} scenes")
    for _, row in runner_scene_manifest.iterrows():
        batch_index = int(row["batch_index"])
        status_path = task_status_path(row)
        if status_path.exists():
            print(f"[{batch_index}/{len(scene_manifest)}] skip: status_exists ({row['date']} {row['landsat_product_id']})")
            continue

        started = datetime.now(ZoneInfo("Asia/Seoul"))
        payload = scene_payload(row)
        group = str(row["period_group"])
        season = str(row["season_group"])
        season_ko = str(row["season_name_ko"])
        run_outputs = {}
        status = "OK"
        error = ""
        mae_k = np.nan
        clear_pixel_count = 0
        print(f"[{batch_index}/{len(scene_manifest)}] {group}/{season}({season_ko}) {payload['date']} {payload['landsat_product_id']} runner={RUNNER_ID_SAFE}")
        try:
            run_outputs = run_scene_in_current_notebook(row)
            run_root = run_outputs.get("run_root", "")
            mae_k, clear_pixel_count = mae_for_run(run_root)
            print("  run root:", run_root)
            print(f"  clear-pixel MAE: {mae_k:.3f} K (n={clear_pixel_count})")
        except Exception as exc:
            status = "ERROR"
            error = f"{type(exc).__name__}: {exc}"
            print("  ERROR:", error)
            if STOP_ON_ERROR:
                raise
        finished = datetime.now(ZoneInfo("Asia/Seoul"))
        row_dict = {
            "task_key": task_key(row),
            "batch_index": batch_index,
            "period_group": group,
            "season_group": season,
            "season_name_ko": season_ko,
            "date": payload["date"],
            "landsat_product_id": payload["landsat_product_id"],
            "scene_id": payload["scene_id"],
            "status": status,
            "error": error,
            "run_root": run_outputs.get("run_root", ""),
            "clear_pixel_count": clear_pixel_count,
            "mae_k": mae_k,
            "runner_id": RUNNER_ID_SAFE,
            "started_kst": started.isoformat(),
            "finished_kst": finished.isoformat(),
        }
        write_task_status(row_dict)
        if status in COMBINED_MANIFEST_STATUSES:
            exec_rows.append(row_dict)
            write_runner_execution_manifest(exec_rows)
    print("runner execution manifest:", RUNNER_EXECUTION_MANIFEST_CSV)
    print("task status dir:", TASK_STATUS_DIR)


In [ ]:
# 평균 MAE 계산: 생성 완료 후 실행
# 관측 가능한 clear pixel에서 model prediction과 Landsat 원본 LST를 비교한다.
from pathlib import Path
import shutil
import os
import json
import numpy as np
import pandas as pd


def _scalar_from_npz(z, key, default=""):
    if key not in z.files:
        return default
    arr = np.asarray(z[key])
    return arr.item() if arr.shape == () else arr


def _status_records():
    records = []
    status_dirs = []
    if "ROI_BATCH_DIR" in globals() and Path(ROI_BATCH_DIR).exists():
        status_dirs.extend(sorted(Path(ROI_BATCH_DIR).glob("_task_status*")))
        legacy_status_dir = Path(ROI_BATCH_DIR) / "_task_status"
        if legacy_status_dir.exists():
            status_dirs.append(legacy_status_dir)
    if "TASK_STATUS_DIR" in globals() and Path(TASK_STATUS_DIR).exists():
        status_dirs.append(Path(TASK_STATUS_DIR))
    for status_dir in sorted(set(status_dirs)):
        for status_file in sorted(status_dir.glob("*.json")):
            try:
                records.append(json.loads(status_file.read_text(encoding="utf-8")))
            except Exception as exc:
                print(f"warning: failed to read task status {status_file}: {exc}")
    return records


def _reconstruction_npz_paths():
    paths = []
    status_records = _status_records()
    if status_records:
        for item in status_records:
            if str(item.get("status", "")) != "OK":
                continue
            run_root = str(item.get("run_root", ""))
            if not run_root or run_root == "nan":
                continue
            paths.extend(sorted((Path(run_root) / "outputs" / "reconstruction").glob("*_lst_reconst.npz")))
    if not paths and "RUNNER_EXECUTION_MANIFEST_CSV" in globals():
        for manifest_csv in sorted(Path(ROI_BATCH_DIR).glob(f"{ROI_LABEL}_batch_execution_manifest_*.csv")):
            exec_df = pd.read_csv(manifest_csv)
            if "status" in exec_df.columns:
                exec_df = exec_df[exec_df["status"].astype(str) == "OK"].copy()
            for _, item in exec_df.iterrows():
                run_root = str(item.get("run_root", ""))
                if not run_root or run_root == "nan":
                    continue
                paths.extend(sorted((Path(run_root) / "outputs" / "reconstruction").glob("*_lst_reconst.npz")))
    if not paths and "EXECUTION_MANIFEST_CSV" in globals() and Path(EXECUTION_MANIFEST_CSV).exists():
        exec_df = pd.read_csv(EXECUTION_MANIFEST_CSV)
        if "status" in exec_df.columns:
            exec_df = exec_df[exec_df["status"].astype(str) == "OK"].copy()
        for _, item in exec_df.iterrows():
            run_root = str(item.get("run_root", ""))
            if not run_root or run_root == "nan":
                continue
            paths.extend(sorted((Path(run_root) / "outputs" / "reconstruction").glob("*_lst_reconst.npz")))
    if not paths:
        paths = sorted(ROI_BATCH_DIR.glob("*/*/runs/*/outputs/reconstruction/*_lst_reconst.npz"))
    return sorted(set(paths))


def _reconstruction_period_season(npz_path, z):
    period = str(_scalar_from_npz(z, "period_group", ""))
    season = str(_scalar_from_npz(z, "season_group", ""))
    season_ko = str(_scalar_from_npz(z, "season_name_ko", ""))
    if period and period != "nan" and season and season != "nan":
        return period, season, season_ko
    parts = Path(npz_path).parts
    if "runs" in parts:
        run_idx = parts.index("runs")
        if run_idx >= 2:
            period = parts[run_idx - 2]
            season = parts[run_idx - 1]
            season_ko = SEASON_DEFINITIONS.get(season, {}).get("name_ko", season_ko)
    return period, season, season_ko


def _mae_row(npz_path):
    with np.load(npz_path, allow_pickle=True) as z:
        if "prediction" not in z.files or "lst_k_raw_direct" not in z.files or "lst_clear_mask_direct" not in z.files:
            return None
        pred = z["prediction"].astype(np.float32)
        raw = z["lst_k_raw_direct"].astype(np.float32)
        clear = z["lst_clear_mask_direct"].astype(bool)
        pred_lst = pred[0] if pred.ndim == 3 else pred
        mask = clear & np.isfinite(raw) & np.isfinite(pred_lst)
        n = int(mask.sum())
        if n == 0:
            return None
        diff = pred_lst[mask] - raw[mask]
        period, season, season_ko = _reconstruction_period_season(npz_path, z)
        return {
            "npz_path": str(npz_path),
            "run_label": str(_scalar_from_npz(z, "run_label", npz_path.parents[2].name)),
            "roi_label": str(_scalar_from_npz(z, "roi_label", "")),
            "requested_date": str(_scalar_from_npz(z, "requested_date", "")),
            "selected_landsat_date": str(_scalar_from_npz(z, "selected_landsat_date", "")),
            "period_group": period,
            "season_group": season,
            "season_name_ko": season_ko,
            "clear_pixel_count": n,
            "mae_k": float(np.mean(np.abs(diff))),
            "rmse_k": float(np.sqrt(np.mean(diff ** 2))),
            "bias_k": float(np.mean(diff)),
            "abs_error_sum": float(np.sum(np.abs(diff))),
            "squared_error_sum": float(np.sum(diff ** 2)),
            "error_sum": float(np.sum(diff)),
        }


def _weighted_summary(frame, group_cols=None):
    if group_cols is None:
        groups = [((), frame)]
    else:
        groups = frame.groupby(group_cols, dropna=False)
    rows = []
    for key, group in groups:
        if group_cols is None:
            row = {"period_group": "ALL", "season_group": "ALL", "season_name_ko": "전체"}
        else:
            if not isinstance(key, tuple):
                key = (key,)
            row = dict(zip(group_cols, key))
        pixel_count = float(group["clear_pixel_count"].sum())
        row.update({
            "sample_count": int(len(group)),
            "clear_pixel_count": int(pixel_count),
            "mae_k": float(group["abs_error_sum"].sum() / pixel_count),
            "rmse_k": float(np.sqrt(group["squared_error_sum"].sum() / pixel_count)),
            "bias_k": float(group["error_sum"].sum() / pixel_count),
            "scene_mean_mae_k": float(group["mae_k"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows)


mae_rows = []
for npz_path in _reconstruction_npz_paths():
    row = _mae_row(npz_path)
    if row is not None:
        mae_rows.append(row)

if not mae_rows:
    raise RuntimeError("MAE를 계산할 reconstruction npz가 없습니다. 배치 생성 후 다시 실행하세요.")

mae_df = pd.DataFrame(mae_rows)
overall = _weighted_summary(mae_df)
by_season = _weighted_summary(mae_df, ["period_group", "season_group", "season_name_ko"])
summary = pd.concat([overall, by_season], ignore_index=True)

MAE_DETAIL_CSV = ROI_BATCH_DIR / f"{ROI_LABEL}_reconstruction_mae_detail.csv"
MAE_SUMMARY_CSV = ROI_BATCH_DIR / f"{ROI_LABEL}_reconstruction_mae_summary.csv"
mae_df.to_csv(MAE_DETAIL_CSV, index=False, encoding="utf-8-sig")
summary.to_csv(MAE_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("MAE detail:", MAE_DETAIL_CSV)
print("MAE summary:", MAE_SUMMARY_CSV)
print("평균 MAE 요약:")
print(summary.to_string(index=False))
